# 📋 Project TODO & Results Log

*Edit this cell directly as you go — check off `[ ]` → `[x]`, fill in numbers, add notes.
This is the running lab notebook for the whole project, kept in one place across
Kaggle/Colab/laptop/4090 sessions.*

---

## To-do

### Setup & data (no GPU)
- [ ] GitHub repo created for the notebook/code; `.gitignore` set up (hf_cache/, checkpoints/, *.jsonl, .env)
- [ ] `HF_ARTIFACT_REPO` set (§0) for checkpoint/artifact sync; `HF_TOKEN` has write access
- [ ] Run notebook top to bottom once this session (defines everything, caches downloads)
- [ ] `sync_pull_artifacts()` run at session start (§0)
- [ ] `run_offline_self_tests()` passes (§28)
- [ ] §15 end-to-end demo runs on a real image

### Aim 1: training data (API only, no GPU)
- [ ] `GEMINI_API_KEYS` set (comma-separated) — each key confirmed to be a SEPARATE Google Cloud project (check AI Studio); `ANTHROPIC_API_KEY` set for the verifier
- [ ] `GEMINI_RPM_LIMIT` / `GEMINI_RPD_LIMIT` (§16) adjusted to match AI Studio's live numbers for your actual model/project, not left at the placeholder defaults
- [ ] §16 pilot run (2 images) — verified rate: ____%
- [ ] `estimate_aim1_api_calls()` run — estimated calls: ____ / estimated cost: $____
- [ ] `run_aim1_batch()` full run — images processed: ____ / verified traces: ____ (across ____ day(s), if it hit a daily cap and resumed)

### Baselines
- [ ] Zero-shot VLM baseline (§20) run on held-out sample
- [ ] Majority-class baseline (§20) computed
- [ ] Diagnosis-baseline detector (§30) trained + evaluated

### Stage 0: grounding detector (§24)
- [ ] Sanity check (`subset_n=20`) trains without error
- [ ] Scaled-up training run
- [ ] `evaluate_stage0_detector()` on holdout — precision / recall / F1: ____ / ____ / ____
- [ ] Visually checked on ≥5 held-out images — trustworthy? Y / N

### SFT (§17)
- [ ] Ran on the full Aim-1 dataset
- [ ] Checkpoint saved — tag: ____________
- [ ] Format-compliance rate before → after SFT: ____% → ____%

### GRPO (§18)
- [ ] Smoke test passed (`validate_span_alignment` = True; ratio moves off 1 at `epochs_per_batch=2`)
- [ ] Real training run (`group_size=___`, steps=___) on the 4090
- [ ] Repeated with ≥2 more random seeds — reward spread across seeds: ____
- [ ] Checkpoint saved — tag: ____________
- [ ] Training curve plotted — reward trending up? Y / N

### Evaluation (§19-23, 26)
- [ ] `run_full_evaluation_suite()` on the full held-out set
- [ ] H1 ablation (tools vs. no tools) — result + 95% CI: ____
- [ ] H2 (checkpoint comparison: base vs. SFT vs. GRPO) — results: ____
- [ ] R_judge grounding rate: ____%
- [ ] Reward-weight sweep (§21) — best weighting found: ____
- [ ] Failure-mode breakdown run
- [ ] Cross-dataset generalization (§27) — once `SECOND_DATASET_PATH` is found

### Analysis & writing
- [ ] 2-4 qualitative examples selected (success + failure) for the paper
- [ ] Related-work section finalized with real results woven in
- [ ] Results section written with actual numbers (pull from the table below)
- [ ] Limitations section written
- [ ] Target venue chosen + current page limit/format/deadline checked
- [ ] Code/data release prepped (README, requirements.txt, API keys stripped)

---

## Results log

*Paste real numbers here as they come in — this becomes the results table.*

| Date | Condition | n | FDI acc | Balanced acc | Format % | Mean reward | Notes |
|---|---|---|---|---|---|---|---|
| | zero-shot VLM | | | | | | |
| | majority baseline | | | | | | |
| | SFT only | | | | | | |
| | full agent (SFT+GRPO) | | | | | | |
| | no-tools ablation | | | | | | |
| | diagnosis-baseline detector | | | | | | |

---

## Notes / decisions log

*Anything decided along the way that future-you should remember.*

-


# DENTEX Agentic VLM — Starter Notebook

Companion to: `agentic-orthodontic-vlm-research-landscape.md` and `dentex-agentic-vlm-proposal.md`

This is the **Phase 1 / Tier 1** notebook from the proposal's timeline: it runs on Kaggle, Colab
free tier, or a laptop GPU (no 4090 needed yet). It covers, in order:

1. **Environment & persistent storage** (0) so datasets/models/checkpoints are downloaded once.
2. **DENTEX loading** (1), **data quality checks** (2), **examples** (3), **preprocessing** (4),
   and **statistics/histograms** (5), including a hierarchy overview and parquet caching.
3. **Tool suite** (6, 11, 12): zoom/crop, contrast enhancement, FDI numbering, and a temporary
   **oracle grounding tool** standing in for Stage 0's trained detector — plus an offline smoke
   test needing no internet/dataset/GPU at all.
4. **The VLM backbone** (7): Qwen2.5-VL-3B-Instruct, 4-bit quantized, with a vision-token probe
   and a rough GRPO memory/rollout-budget estimate for your actual hardware tiers.
5. **Sanity check, checkpoints, reproducibility** (8, 9, 10): one quick inference test, save/load
   helpers for every later training stage, and a persistent log of environment + package versions
   across Kaggle/Colab/laptop/4090.
6. **The agent orchestration loop** (13): the actual prompt→generate→parse→tool-call-or-answer
   loop from §5.3/§5.4, run end to end on a real DENTEX image with the loaded model.
7. **The composite reward function** (14) from §5.5, scored on a real trajectory (15).
8. **The Aim 1 trace generation + verification pipeline** (16, §5.2): a two-model
   generator/verifier setup over LLM APIs — needs no GPU, so it's runnable independently of
   everything else here if you have API keys handy.
9. **Stage 1 SFT** (17): a manual LoRA/QLoRA training loop on the verified traces.
10. **Stage 2 GRPO** (18): a from-scratch reference implementation of group rollouts → graded
    reward → group-normalized advantage → policy update, with a built-in check
    (`validate_span_alignment`) for its trickiest assumption before you trust it.

> First full run will take a while (dataset ≈ a few GB, model ≈ a few GB). Every run after that
> should be much faster, since both are cached under `PERSIST_DIR` instead of `/tmp`. Section 6's
> smoke test and section 16's pilot trace generation can each be run largely standalone.
>
> **Honest expectation-setting**: sections 13/15 run the *base* (not yet fine-tuned) model
> through the agent loop. Expect mostly malformed or ignored tool calls at this stage — that's
> normal, not a bug, and is exactly the gap Stage 1 SFT closes. Section 18's GRPO now includes
> a KL penalty and PPO-style clipping, but is still a deliberately simplified reference
> implementation meant for understanding the mechanics at small scale, not a production
> trainer — see its own header for what to validate before trusting it.


## 0. Environment detection & persistent storage

In [ ]:
import os
import sys


def detect_environment():
    """Detect whether we're running on Kaggle, Colab, or locally, and pick a
    persistent storage directory accordingly."""
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle"):
        return "kaggle"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    return "local"


ENV = detect_environment()
print(f"Detected environment: {ENV}")

if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_DIR = "/content/drive/MyDrive/orthodontic_agent"
    print("Using Google Drive for persistent storage (survives across Colab sessions).")

elif ENV == "kaggle":
    PERSIST_DIR = "/kaggle/working/orthodontic_agent"
    print(
        "Using /kaggle/working for this session's storage.\n"
        "NOTE: /kaggle/working only reliably persists within the current session/version.\n"
        "To avoid re-downloading in a brand-new session, use 'Save Version' to commit this\n"
        "folder as output, or save it as a Kaggle Dataset and attach it as an input next time."
    )

else:
    PERSIST_DIR = os.path.expanduser("~/orthodontic_agent_cache")
    print(f"Using local directory for persistent storage: {PERSIST_DIR}")

os.makedirs(PERSIST_DIR, exist_ok=True)
print(f"PERSIST_DIR = {PERSIST_DIR}")


In [ ]:
# Safe to re-run: pip skips packages that are already installed and up to date.
!pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub \
    qwen-vl-utils pillow matplotlib pandas pyarrow tqdm openai anthropic google-genai torchvision scikit-learn tabulate


In [ ]:
# IMPORTANT: these env vars must be set BEFORE the first import of
# transformers / datasets / huggingface_hub, or those libraries will already have
# picked a default (non-persistent) cache location.
HF_CACHE_DIR = os.path.join(PERSIST_DIR, "hf_cache")
CHECKPOINT_DIR = os.path.join(PERSIST_DIR, "checkpoints")
DATA_DIR = os.path.join(PERSIST_DIR, "data")

for d in (HF_CACHE_DIR, CHECKPOINT_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE_DIR, "datasets")
# If you have a Hugging Face token (not required for DENTEX or Qwen2.5-VL, but useful
# to raise API rate limits), uncomment and set it:
# os.environ["HF_TOKEN"] = "hf_..."

print("Downloads will be cached under:", HF_CACHE_DIR)
print("Re-running this notebook later will reuse the cache instead of re-downloading.")


In [ ]:
import json
import glob
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

import torch
from huggingface_hub import snapshot_download

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
CONFIG = {
    "dataset_repo": "ibrahimhamamci/DENTEX",
    # Staged choice from the proposal: 3B for Kaggle/Colab/laptop prototyping,
    # promoted to Qwen2.5-VL-7B-Instruct once RTX 4090 / server access is available.
    "model_name": "Qwen/Qwen2.5-VL-3B-Instruct",
    "seed": 42,
}
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
print(CONFIG)


### Syncing across sessions: two different problems, two different tools

Moving between Kaggle/Colab/laptop/4090 raises two genuinely different sync problems, and
one tool can't solve both:

**1. The notebook's own code** — GitHub, but "syncing" here really means *get the latest
`.ipynb` file, then open that file as your working notebook*, not "hot-reload my
already-running kernel." No platform (Kaggle, Colab, plain Jupyter) can live-reload a
running kernel's cells from an external git pull — that's not a limitation of this
notebook, it's just not how any of them work.

- **Colab** is the easiest case: File → Open notebook → GitHub tab, paste the repo URL,
  pick the `.ipynb`. To push changes back: File → Save a copy in GitHub. No git commands
  needed.
- **Kaggle** has no equivalent native integration for the notebook's own source. Practical
  options: (a) manually download the `.ipynb` from GitHub and use Kaggle's Upload Notebook
  to update your kernel, then manually download+push your changes back at the end of a
  session, or (b) the Kaggle API (`kaggle kernels push`) if you want this scripted — more
  setup, worth it only if you do this often.
- **Local laptop**: the simple case — plain `git clone` / `git pull` / `git add` /
  `git commit` / `git push` in a terminal.
- **Whichever platform**: never commit checkpoints, `hf_cache/`, or any API token to the
  repo. A `.gitignore` with at least `hf_cache/`, `checkpoints/`, `data/*.parquet`,
  `*.jsonl`, `__pycache__/`, `.env` keeps the repo to just code and small config.

**2. Checkpoints + derived artifacts (traces, eval results, logs)** — Hugging Face Hub, not
git. This part genuinely IS automatable from inside a running session, since it's just data
files, not the notebook's own source — the two functions below handle it, reusing the same
`huggingface_hub` this notebook already depends on. Deliberately does **not** sync
`hf_cache/` (the DENTEX download + model weights) — those re-download from their own
canonical source fine in any fresh environment, and syncing many-GB of redundant cache
across sessions would be pure waste.


In [ ]:
HF_ARTIFACT_REPO = None  # <- fill in: a private HF Hub dataset repo you own, e.g. "yourname/dentex-agent-artifacts"


def sync_pull_artifacts():
    """Pull checkpoints + derived artifacts from HF_ARTIFACT_REPO into PERSIST_DIR. Run
    this near the start of a new session, right after PERSIST_DIR is set up -- otherwise an
    empty local checkpoints/ folder looks like 'no prior work exists' when it might just
    mean 'not pulled yet'."""
    if not HF_ARTIFACT_REPO:
        print("Set HF_ARTIFACT_REPO above first (a private HF Hub dataset repo you own).")
        return
    try:
        snapshot_download(
            repo_id=HF_ARTIFACT_REPO, repo_type="dataset", local_dir=PERSIST_DIR,
            allow_patterns=["checkpoints/**", "data/**", "run_log.jsonl"],
        )
        print(f"Pulled artifacts from {HF_ARTIFACT_REPO} into {PERSIST_DIR}.")
    except Exception as e:
        print(f"Nothing to pull yet, or pull failed ({e}) -- expected and fine on a "
              f"first-ever run before anything has been pushed.")


def sync_push_artifacts(commit_message="sync from notebook session"):
    """Push checkpoints + derived artifacts (NOT hf_cache/) to HF_ARTIFACT_REPO. Needs an
    HF_TOKEN with WRITE access -- a read-only token (all that section 0's original comment
    needed) will fail here; generate a new one with write scope at
    huggingface.co/settings/tokens. Call this whenever you want the current session's
    progress backed up: after a checkpoint save, or before a Kaggle/Colab session might
    time out."""
    if not HF_ARTIFACT_REPO:
        print("Set HF_ARTIFACT_REPO above first.")
        return
    from huggingface_hub import HfApi, create_repo
    try:
        create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
        HfApi().upload_folder(
            repo_id=HF_ARTIFACT_REPO, repo_type="dataset", folder_path=PERSIST_DIR,
            allow_patterns=["checkpoints/**", "data/**", "run_log.jsonl"],
            commit_message=commit_message,
        )
        print(f"Pushed checkpoints + artifacts to {HF_ARTIFACT_REPO}.")
    except Exception as e:
        print(f"Push failed: {e}. Check HF_TOKEN has write access to this repo.")


print("Set HF_ARTIFACT_REPO above, then:\n"
      "  sync_pull_artifacts()   # near the start of a session\n"
      "  sync_push_artifacts()   # whenever this session's progress should be backed up\n"
      "  (save_checkpoint() in section 9 still only saves locally -- call "
      "sync_push_artifacts() afterward if you want it backed up immediately, not just at "
      "the end of the session.)")


## 1. Download and load the DENTEX dataset

In [ ]:
# snapshot_download reuses files already present in the cache on subsequent runs --
# it does NOT re-download everything from scratch every time you run this cell.
dentex_path = snapshot_download(
    repo_id=CONFIG["dataset_repo"],
    repo_type="dataset",
    cache_dir=os.environ["HF_HUB_CACHE"],
)
print("DENTEX dataset files are available at:", dentex_path)


In [ ]:
# The DENTEX repo mixes an imagefolder-style layout for some splits with plain JSON
# annotation files for others, so `datasets.load_dataset(...)` can't auto-infer a single
# format across all splits (this is a known quirk of this specific repo, not a bug in
# your setup). We inspect the raw files ourselves instead of relying on auto-loading.
def list_files(root, max_depth=3, max_per_dir=15):
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > max_depth:
            dirnames[:] = []
            continue
        indent = "  " * depth
        print(f"{indent}{Path(dirpath).name}/")
        for f in sorted(filenames)[:max_per_dir]:
            print(f"{indent}  {f}")
        if len(filenames) > max_per_dir:
            print(f"{indent}  ... ({len(filenames) - max_per_dir} more files)")

list_files(dentex_path)


In [ ]:
def load_coco_json(path):
    with open(path) as f:
        return json.load(f)

json_files = sorted(glob.glob(os.path.join(dentex_path, "**", "*.json"), recursive=True))
print(f"Found {len(json_files)} JSON file(s).")

all_coco = {}
for jf in json_files:
    try:
        all_coco[jf] = load_coco_json(jf)
    except Exception as e:
        print(f"Could not parse {jf}: {e}")


### Hierarchy overview

DENTEX provides three hierarchically annotated sets (quadrant-only, quadrant-enumeration,
and the fully-labeled quadrant-enumeration-diagnosis set used for the actual benchmark), plus
train/val/test splits. This table lays all discovered annotation files out side by side so
it's clear which is which before picking one to work with.


In [ ]:
summary_rows = []
for jf, d in all_coco.items():
    if not isinstance(d, dict):
        continue
    summary_rows.append({
        "file": os.path.relpath(jf, dentex_path),
        "n_images": len(d.get("images", [])),
        "n_annotations": len(d.get("annotations", [])),
        "n_categories": len(d.get("categories", [])),
        "category_names": ", ".join(c.get("name", "?") for c in d.get("categories", [])[:8]),
    })

hierarchy_summary = pd.DataFrame(summary_rows)
print("Overview of all annotation files found (the 3 DENTEX hierarchy levels x splits):")
hierarchy_summary


### Pick the fully-annotated (quadrant-enumeration-diagnosis) train file

The cell below picks automatically using a heuristic (train split + most category fields
per annotation + "diagnosis"/"disease" in the filename). **If it looks wrong** given the
table above, set `best_path` manually instead, e.g.:
```python
best_path = json_files[<index>]
coco = all_coco[best_path]
```


In [ ]:
candidates = [(jf, d) for jf, d in all_coco.items() if isinstance(d, dict) and d.get("annotations")]

if not candidates:
    raise RuntimeError(
        "No COCO-style JSON with an 'annotations' field was found. "
        "Re-check the hierarchy table above and adjust the search."
    )

def score_candidate(jf, d):
    score = 0
    name = jf.lower()
    if "train" in name:
        score += 1
    ann0 = d["annotations"][0] if d["annotations"] else {}
    for key in ("category_id_1", "category_id_2", "category_id_3", "extra"):
        if key in ann0:
            score += 1
    if "diagnosis" in name or "disease" in name:
        score += 2
    return score

candidates.sort(key=lambda c: score_candidate(*c), reverse=True)
best_path, coco = candidates[0]

print("Selected annotation file:", best_path)
print("Number of images:", len(coco.get("images", [])))
print("Number of annotations:", len(coco.get("annotations", [])))
print("Categories:", coco.get("categories", [])[:20])


### Parse into DataFrames (cached to persistent storage)

The parsed DataFrames are cached to parquet under `DATA_DIR`. Every run after the first
loads straight from the cache instead of re-parsing the JSON.


In [ ]:
cache_images_pq = os.path.join(DATA_DIR, "images_df.parquet")
cache_annots_pq = os.path.join(DATA_DIR, "annots_df.parquet")
cache_categories_pq = os.path.join(DATA_DIR, "categories_df.parquet")

if all(os.path.exists(p) for p in (cache_images_pq, cache_annots_pq, cache_categories_pq)):
    print("Loading cached parsed DataFrames from persistent storage (skipping JSON re-parse)...")
    images_df = pd.read_parquet(cache_images_pq)
    annots_df = pd.read_parquet(cache_annots_pq)
    categories_df = pd.read_parquet(cache_categories_pq)
else:
    images_df = pd.DataFrame(coco["images"])
    annots_df = pd.DataFrame(coco["annotations"])
    categories_df = pd.DataFrame(coco.get("categories", []))
    annots_df["bbox"] = annots_df["bbox"].apply(list)  # ensure a plain list column for parquet
    images_df.to_parquet(cache_images_pq)
    annots_df.to_parquet(cache_annots_pq)
    categories_df.to_parquet(cache_categories_pq)
    print("Parsed DataFrames built and cached to:", DATA_DIR)

print("images_df columns:", list(images_df.columns))
print("annots_df columns:", list(annots_df.columns))
print("\ncategories_df:")
print(categories_df)

images_df.head()


In [ ]:
annots_df.head()


## 2. Data quality checks

In [ ]:
# Map each DENTEX image record to the actual image file found on disk. Needed for
# every downstream step (quality checks, visualization, preprocessing, inference).
image_files = (
    glob.glob(os.path.join(dentex_path, "**", "*.png"), recursive=True)
    + glob.glob(os.path.join(dentex_path, "**", "*.jpg"), recursive=True)
)
print(f"Found {len(image_files)} image files on disk.")

by_basename = {os.path.basename(p): p for p in image_files}
images_df["local_path"] = images_df["file_name"].apply(
    lambda fn: by_basename.get(os.path.basename(fn))
)
missing = images_df["local_path"].isna().sum()
print(f"{missing} / {len(images_df)} images could not be matched to a file on disk.")
print("(If this number is high, re-check the hierarchy/directory listing above -- the "
      "image subfolder name may differ from what this cell assumed.)")


In [ ]:
# Corrupt/unreadable image check -- better to catch this in five minutes here than
# hours into a Kaggle session when a training job crashes on a bad file.
bad_files = []
modes = []
for p in images_df["local_path"].dropna():
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            modes.append(im.mode)
    except Exception as e:
        bad_files.append((p, str(e)))

print(f"Checked {images_df['local_path'].notna().sum()} images.")
print(f"Corrupt/unreadable: {len(bad_files)}")
for p, err in bad_files[:10]:
    print(" -", p, "->", err)

mode_counts = pd.Series(modes).value_counts()
print("\nImage color-mode distribution:")
print(mode_counts)
print(
    "\nIf 'L' (grayscale) dominates, that confirms the RGB-conversion step in "
    "preprocess_image() below is necessary, not just defensive."
)


In [ ]:
# Bounding-box sanity check: every box should lie within its image's dimensions.
if "width" in images_df.columns and "height" in images_df.columns:
    dims = images_df.set_index("id")[["width", "height"]]
    merged = annots_df.join(dims, on="image_id")

    def box_out_of_bounds(row):
        x, y, w, h = row["bbox"]
        return x < 0 or y < 0 or x + w > row["width"] or y + h > row["height"]

    oob_mask = merged.apply(box_out_of_bounds, axis=1)
    print(f"Bounding boxes outside image bounds: {oob_mask.sum()} / {len(merged)}")
    if oob_mask.sum():
        print(merged[oob_mask][["image_id", "bbox", "width", "height"]].head())
else:
    print("images_df has no width/height columns -- derive them from the actual image "
          "files first if you want this check to run.")


### Held-out evaluation split

The evaluation harness later in this notebook (sections 19-22) must score checkpoints on
images that were never used for Aim-1 trace generation, SFT, or GRPO — otherwise every
metric is measuring memorization, not generalization. This tries DENTEX's own official test
split first; many benchmark test sets ship without public ground truth, so it falls back to
carving a fixed-seed held-out slice out of the train split if that's the case here, and says
plainly which one it's actually using. `images_df`/`annots_df` keep every image (including the
held-out ones) — `run_agent()` needs to look up any image by id — but `holdout_ids` below is
the actual boundary: anything that *generates training data* (Aim-1, SFT, GRPO image
selection) must exclude it; anything that *evaluates* (sections 19-22) must sample only from it.


In [ ]:
def _score_named_split(jf, d, split_name):
    score = int(split_name in jf.lower())
    ann0 = d["annotations"][0] if d["annotations"] else {}
    score += sum(k in ann0 for k in ("category_id_1", "category_id_2", "category_id_3", "extra"))
    score += 2 * int("diagnosis" in jf.lower() or "disease" in jf.lower())
    return score

_annotated = [(jf, d) for jf, d in all_coco.items() if isinstance(d, dict) and d.get("annotations")]
_test_ranked = sorted(_annotated, key=lambda c: _score_named_split(*c, "test"), reverse=True)

if _test_ranked and "test" in _test_ranked[0][0].lower() and _test_ranked[0][0] != best_path:
    EVAL_SOURCE = "official DENTEX test split (merged into images_df/annots_df below)"
    _, _test_coco = _test_ranked[0]
    _test_images_df = pd.DataFrame(_test_coco["images"])
    _test_images_df["local_path"] = _test_images_df["file_name"].apply(
        lambda fn: by_basename.get(os.path.basename(fn))
    )
    _test_annots_df = pd.DataFrame(_test_coco["annotations"])
    _test_annots_df["bbox"] = _test_annots_df["bbox"].apply(list)

    holdout_ids = set(_test_images_df["id"])
    images_df = pd.concat([images_df, _test_images_df], ignore_index=True)
    annots_df = pd.concat([annots_df, _test_annots_df], ignore_index=True)
else:
    EVAL_SOURCE = "held-out slice of the TRAIN split (fixed seed) -- NOT DENTEX's official test set"
    _rng = np.random.default_rng(CONFIG["seed"])
    _all_ids = images_df["id"].unique()
    holdout_ids = set(_rng.choice(_all_ids, size=int(len(_all_ids) * 0.2), replace=False))

print(f"Evaluation source: {EVAL_SOURCE}")
print(f"Held-out evaluation ids: {len(holdout_ids)}")
print(f"Remaining pool for training/dev use: {len(set(images_df['id']) - holdout_ids)} images")
print(
    "\nFrom here on: sample FROM `holdout_ids` when evaluating (sections 19-22); sample "
    "EXCLUDING `holdout_ids` when generating training data (Aim-1 §16, SFT §17, GRPO §18)."
)


## 3. Show a couple of examples

In [ ]:
def draw_annotations(image_path, annots_for_image, categories_df):
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    cat_lookup = (
        dict(zip(categories_df["id"], categories_df["name"])) if len(categories_df) else {}
    )
    for _, ann in annots_for_image.iterrows():
        x, y, w, h = ann["bbox"]
        draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
        label_parts = []
        for cat_col in ("category_id_1", "category_id_2", "category_id_3", "category_id"):
            if cat_col in ann and pd.notna(ann[cat_col]):
                label_parts.append(str(cat_lookup.get(ann[cat_col], ann[cat_col])))
        if label_parts:
            draw.text((x, max(0, y - 12)), "/".join(label_parts), fill="red")
    return img


available = images_df.dropna(subset=["local_path"])
sample_image_ids = available["id"].sample(
    min(3, len(available)), random_state=CONFIG["seed"]
).tolist()

fig, axes = plt.subplots(1, len(sample_image_ids), figsize=(6 * len(sample_image_ids), 6))
if len(sample_image_ids) == 1:
    axes = [axes]

for ax, img_id in zip(axes, sample_image_ids):
    row = images_df[images_df["id"] == img_id].iloc[0]
    ann_rows = annots_df[annots_df["image_id"] == img_id]
    vis = draw_annotations(row["local_path"], ann_rows, categories_df)
    ax.imshow(vis)
    ax.set_title(f"image_id={img_id}  ({len(ann_rows)} annotated teeth)")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 4. Basic preprocessing

In [ ]:
def preprocess_image(img: Image.Image, target_size=(1024, 512)):
    """
    Basic preprocessing for panoramic dental X-rays:
      1. Convert to RGB (panoramic X-rays are often single-channel; the VLM's vision
         encoder expects 3-channel input).
      2. Resize with aspect ratio preserved, then pad to a fixed target size (panoramic
         X-rays are wide and short -- a plain square resize distorts tooth shape, which
         matters for a model that has to reason about anatomy).
    Returns the processed image plus the scale/offset needed to remap bounding boxes
    into the new coordinate space later (useful for the zoom/crop tool and SFT data).
    """
    img = img.convert("RGB")
    target_w, target_h = target_size
    scale = min(target_w / img.width, target_h / img.height)
    new_w, new_h = int(img.width * scale), int(img.height * scale)
    resized = img.resize((new_w, new_h), Image.BICUBIC)

    padded = Image.new("RGB", target_size, (0, 0, 0))
    paste_x = (target_w - new_w) // 2
    paste_y = (target_h - new_h) // 2
    padded.paste(resized, (paste_x, paste_y))
    return padded, scale, (paste_x, paste_y)


demo_row = images_df.dropna(subset=["local_path"]).iloc[0]
original = Image.open(demo_row["local_path"])
processed, scale, offset = preprocess_image(original)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(original, cmap="gray" if original.mode == "L" else None)
axes[0].set_title(f"Original  {original.size}  mode={original.mode}")
axes[0].axis("off")
axes[1].imshow(processed)
axes[1].set_title(f"Preprocessed  {processed.size}  (scale={scale:.3f})")
axes[1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
def remap_bbox(bbox, scale, offset):
    """Remap a [x, y, w, h] bounding box from original image coordinates into the
    coordinate space produced by preprocess_image() (same scale + padding offset).
    Needed to keep ground-truth boxes aligned once images are resized/padded --
    e.g. when building the SFT trace dataset or the zoom/crop tool's inputs."""
    x, y, w, h = bbox
    off_x, off_y = offset
    return [x * scale + off_x, y * scale + off_y, w * scale, h * scale]


demo_ann = annots_df[annots_df["image_id"] == demo_row["id"]]
if len(demo_ann):
    orig_bbox = demo_ann.iloc[0]["bbox"]
    remapped = remap_bbox(orig_bbox, scale, offset)
    print(f"Original bbox:  {orig_bbox}")
    print(f"Remapped bbox:  {[round(v, 1) for v in remapped]}  (in the {processed.size} preprocessed image)")


## 5. Dataset statistics and histograms

In [ ]:
if "width" in images_df.columns and "height" in images_df.columns:
    widths, heights = images_df["width"], images_df["height"]
else:
    sizes = images_df.dropna(subset=["local_path"])["local_path"].apply(lambda p: Image.open(p).size)
    widths, heights = zip(*sizes)
    widths, heights = pd.Series(widths), pd.Series(heights)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=30)
axes[0].set_title("Image width distribution")
axes[0].set_xlabel("pixels")
axes[1].hist(heights, bins=30)
axes[1].set_title("Image height distribution")
axes[1].set_xlabel("pixels")
plt.tight_layout()
plt.show()

print(f"Width:  min={widths.min()}  max={widths.max()}  mean={widths.mean():.0f}")
print(f"Height: min={heights.min()}  max={heights.max()}  mean={heights.mean():.0f}")
print("DENTEX combines 3 institutions with different equipment, so real spread here is "
      "expected -- this is exactly why the fixed-aspect-ratio preprocessing above matters.")


In [ ]:
diag_col = next((c for c in ("category_id_3", "category_id") if c in annots_df.columns), None)

if diag_col and len(categories_df):
    cat_lookup = dict(zip(categories_df["id"], categories_df["name"]))
    diag_counts = annots_df[diag_col].map(cat_lookup).value_counts()
    plt.figure(figsize=(8, 4))
    diag_counts.plot(kind="bar")
    plt.title("Diagnosis class distribution (annotated abnormal teeth)")
    plt.ylabel("count")
    plt.tight_layout()
    plt.show()
    print(diag_counts)
    print(
        "\nClass imbalance here directly informs the graded-reward design in the "
        "proposal (partial credit for quadrant/enumeration vs. full credit requiring "
        "the correct, possibly rare, diagnosis class)."
    )
else:
    print("Could not auto-detect the diagnosis category column -- inspect annots_df.columns "
          "and categories_df printed earlier and set diag_col manually.")


In [ ]:
bbox_areas = annots_df["bbox"].apply(lambda b: b[2] * b[3])

plt.figure(figsize=(8, 4))
plt.hist(bbox_areas, bins=40)
plt.title("Annotated tooth/lesion bounding-box area (pixels\u00b2)")
plt.xlabel("area")
plt.tight_layout()
plt.show()

if "width" in images_df.columns and "height" in images_df.columns:
    img_area_lookup = (images_df.set_index("id")["width"] * images_df.set_index("id")["height"])
    rel_area = (bbox_areas.values / annots_df["image_id"].map(img_area_lookup).values)
    rel_area = pd.Series(rel_area).dropna()

    plt.figure(figsize=(8, 4))
    plt.hist(rel_area, bins=40)
    plt.title("Bounding-box area as a fraction of full image area")
    plt.xlabel("fraction")
    plt.tight_layout()
    plt.show()

    print(f"Median annotated box covers {rel_area.median()*100:.2f}% of the full image.")
    print(
        "This is the quantitative case for the zoom/crop tool: most findings occupy a "
        "small fraction of a large panoramic image -- exactly the resolution problem "
        "hypothesis H1 in the proposal argues tool use should address."
    )


In [ ]:
per_image_counts = annots_df.groupby("image_id").size()
plt.figure(figsize=(8, 4))
plt.hist(per_image_counts, bins=range(1, per_image_counts.max() + 2))
plt.title("Number of annotated abnormal teeth per image")
plt.xlabel("count")
plt.tight_layout()
plt.show()
print(per_image_counts.describe())


In [ ]:
# Quadrant x diagnosis co-occurrence -- which quadrants tend to have which
# abnormalities. Useful context for the reward design (§5.5) and for the FDI
# numbering tool below.
quad_col = "category_id_1" if "category_id_1" in annots_df.columns else None

if quad_col and diag_col and len(categories_df):
    cat_lookup = dict(zip(categories_df["id"], categories_df["name"]))
    pivot = pd.crosstab(
        annots_df[quad_col].map(cat_lookup),
        annots_df[diag_col].map(cat_lookup),
    )
    plt.figure(figsize=(8, 5))
    plt.imshow(pivot.values, cmap="Blues", aspect="auto")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha="right")
    plt.yticks(range(len(pivot.index)), pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            plt.text(j, i, pivot.values[i, j], ha="center", va="center")
    plt.title("Quadrant x diagnosis co-occurrence")
    plt.colorbar(label="count")
    plt.tight_layout()
    plt.show()
    print(pivot)
else:
    print("Could not find both a quadrant column and a diagnosis column to cross-tabulate -- "
          "inspect annots_df.columns and categories_df and set quad_col/diag_col manually.")


## 6. Tool suite prototypes (+ offline smoke test)

Two of the proposal's four core tools (§5.3), implemented as plain deterministic functions --
no model weights, no learned components, fully auditable. The smoke test right after them uses
a synthetic dummy image and needs **no internet, no dataset download, and no GPU** -- if you
want to sanity-check this logic before the full DENTEX download/model load finishes (or before
you're back at a machine that can run the rest of this notebook), this cell alone is enough.


In [ ]:
def tool_zoom_crop(image: Image.Image, bbox, padding_frac=0.25):
    """Return a cropped, zoomed-in view around `bbox` (a [x, y, w, h] box), with
    `padding_frac` extra context on each side. This is the agent's zoom/crop tool --
    deterministic, so its behavior is fully predictable and auditable."""
    x, y, w, h = bbox
    pad_x, pad_y = w * padding_frac, h * padding_frac
    left = max(0, x - pad_x)
    top = max(0, y - pad_y)
    right = min(image.width, x + w + pad_x)
    bottom = min(image.height, y + h + pad_y)
    return image.crop((left, top, right, bottom))


def tool_enhance_contrast(image: Image.Image, factor=1.5):
    """Contrast enhancement -- the agent's 'contrast enhancement' tool. Kept as a
    simple, fast, fully deterministic operation so its effect is easy to reason about
    and to unit-test."""
    from PIL import ImageEnhance
    return ImageEnhance.Contrast(image.convert("RGB")).enhance(factor)


def make_synthetic_dental_image(size=(1200, 600)):
    """A cheap synthetic stand-in for a panoramic X-ray -- lets you sanity-check the
    tool functions and preprocessing pipeline right now, with zero internet access and
    zero dependency on the DENTEX download having finished."""
    arr = (np.random.rand(size[1], size[0]) * 60 + 60).astype(np.uint8)  # dark, low-contrast
    arr[250:350, 500:600] = 200  # a brighter rectangular "tooth-like" region to crop/zoom onto
    return Image.fromarray(arr, mode="L")


print("Running offline smoke test (no internet or dataset download required)...")
synth_img = make_synthetic_dental_image()
synth_bbox = [500, 250, 100, 100]

cropped = tool_zoom_crop(synth_img.convert("RGB"), synth_bbox)
enhanced = tool_enhance_contrast(synth_img)
pre, sc, off = preprocess_image(synth_img)
remapped_synth = remap_bbox(synth_bbox, sc, off)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(synth_img, cmap="gray"); axes[0].set_title("synthetic input"); axes[0].axis("off")
axes[1].imshow(cropped); axes[1].set_title("tool_zoom_crop output"); axes[1].axis("off")
axes[2].imshow(enhanced, cmap="gray"); axes[2].set_title("tool_enhance_contrast output"); axes[2].axis("off")
axes[3].imshow(pre); axes[3].set_title(f"preprocess_image\nremapped bbox={[round(v) for v in remapped_synth]}")
axes[3].axis("off")
plt.tight_layout()
plt.show()
print("Smoke test passed if all four panels above rendered without errors.")


## 7. Load the VLM backbone (cached locally after the first run)

In [ ]:
from transformers import BitsAndBytesConfig, AutoProcessor

# Use whichever class name your installed `transformers` version exposes for Qwen2.5-VL.
# Newer versions expose `Qwen2_5_VLForConditionalGeneration`; if that import fails, the
# `Qwen2VLForConditionalGeneration` class (Qwen2-VL) is generally still compatible with
# Qwen2.5-VL checkpoints as a fallback.
try:
    from transformers import Qwen2_5_VLForConditionalGeneration as ModelClass
except ImportError:
    from transformers import Qwen2VLForConditionalGeneration as ModelClass
    print("Using Qwen2VLForConditionalGeneration as a fallback -- consider upgrading "
          "transformers if you want the dedicated Qwen2.5-VL model class.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.time()
processor = AutoProcessor.from_pretrained(CONFIG["model_name"], cache_dir=os.environ["HF_HUB_CACHE"])
model = ModelClass.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=os.environ["HF_HUB_CACHE"],
)
print(f"Loaded {CONFIG['model_name']} in {time.time() - t0:.1f}s "
      f"(subsequent loads reuse the cache and should be much faster).")


In [ ]:
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {n_params/1e9:.2f}B")
print(f"Trainable parameters: {n_trainable/1e9:.4f}B  (0 expected before LoRA adapters are attached)")
print(f"Model dtype (example param): {next(model.parameters()).dtype}")
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    print(f"GPU memory reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")
print("\nVision config:", getattr(model.config, "vision_config", "n/a"))
print("Text num_hidden_layers:", getattr(model.config, "num_hidden_layers", "n/a"))


### Vision-token cost + a rough GRPO compute budget

How many tokens does one DENTEX image actually cost, and what does that imply for the GRPO
group size (G) you can afford on each of your hardware tiers? Treat the numbers below as a
starting point for planning, not a guarantee -- confirm with real `nvidia-smi` /
`torch.cuda.memory_summary()` readings once you're actually training.


In [ ]:
from qwen_vl_utils import process_vision_info

sample_for_tokens = images_df.dropna(subset=["local_path"]).iloc[0]
img_for_tokens = Image.open(sample_for_tokens["local_path"]).convert("RGB")

probe_messages = [{
    "role": "user",
    "content": [{"type": "image", "image": img_for_tokens}, {"type": "text", "text": "describe"}],
}]
probe_text = processor.apply_chat_template(probe_messages, tokenize=False, add_generation_prompt=True)
probe_image_inputs, probe_video_inputs = process_vision_info(probe_messages)
probe_inputs = processor(
    text=[probe_text], images=probe_image_inputs, videos=probe_video_inputs, return_tensors="pt"
)

n_tokens = probe_inputs["input_ids"].shape[1]
print(f"Full prompt (image + template text) token count for one DENTEX image: {n_tokens}")
print(
    "Use this as a rough per-turn token cost when budgeting GRPO group size G: a multi-turn "
    "rollout with k tool calls (each returning a new cropped image) costs roughly k x this "
    "many vision tokens plus the growing text history -- worth capping the tool-call budget "
    "(§5.5 R_efficiency) partly for this reason, not just to discourage redundant calls."
)


In [ ]:
def estimate_grpo_memory_gb(param_count_b, group_size, quant_bits=4, seq_len=2048, dtype_bytes=2):
    """Very rough back-of-envelope estimate -- NOT a substitute for trying it and watching
    nvidia-smi. Assumes QLoRA (base weights quantized; LoRA adapter + optimizer state is small
    relative to base weights; activations scale roughly with group_size and seq_len)."""
    base_weights_gb = param_count_b * 1e9 * (quant_bits / 8) / 1e9
    activations_gb = group_size * seq_len * dtype_bytes * 1e-9 * 50  # crude per-token overhead
    return base_weights_gb, activations_gb, base_weights_gb + activations_gb


for gpu_name, vram in [("Kaggle/Colab T4 or P100", 16), ("RTX 4090", 24)]:
    for g in (4, 8):
        base, act, total = estimate_grpo_memory_gb(3, g, seq_len=n_tokens)
        fits = "fits" if total < vram * 0.8 else "TIGHT / may not fit"
        print(f"{gpu_name:24s}  3B model, G={g}:  ~{total:.1f} GB estimated  ({fits}, budget {vram} GB)")

print(
    "\nTreat these as a starting point for picking G, not a guarantee. Start with a small G "
    "(e.g. 4) on free-tier GPUs and only increase it once you've watched real memory usage."
)


## 8. Quick sanity-check inference

In [ ]:
from qwen_vl_utils import process_vision_info

sample_row = images_df.dropna(subset=["local_path"]).iloc[0]
sample_img = Image.open(sample_row["local_path"]).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": sample_img},
            {"type": "text", "text": (
                "Describe any visible dental abnormalities in this panoramic X-ray, "
                "and note which quadrant they are in."
            )},
        ],
    }
]

text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text_prompt], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
).to(model.device)

t0 = time.time()
with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=256)
gen_trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
output_text = processor.batch_decode(gen_trimmed, skip_special_tokens=True)[0]

print(f"Generated in {time.time() - t0:.1f}s\n")
print(output_text)


## 9. Checkpoint save / load helpers

Call `save_checkpoint(...)` at the end of every future stage (Stage 0 tool fine-tuning,
Stage 1 SFT, Stage 2 GRPO) so a new session can resume from persistent storage instead
of starting over. This saves locally, to `PERSIST_DIR` -- for it to survive a switch to a
*different* machine (Kaggle today, the 4090 tomorrow), also call `sync_push_artifacts()`
from section 0 afterward.


In [ ]:
def save_checkpoint(model, processor, tag):
    """Save the current model/adapter + processor to persistent storage under a named
    tag (e.g. 'sft-3b-v1', 'grpo-3b-step500'). Call this after each training stage."""
    out_dir = os.path.join(CHECKPOINT_DIR, tag)
    os.makedirs(out_dir, exist_ok=True)
    model.save_pretrained(out_dir)
    processor.save_pretrained(out_dir)
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(
            {"tag": tag, "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"), "base_model": CONFIG["model_name"]},
            f, indent=2,
        )
    print(f"Saved checkpoint '{tag}' to {out_dir}")
    return out_dir


def list_checkpoints():
    if not os.path.isdir(CHECKPOINT_DIR):
        return []
    return sorted(os.listdir(CHECKPOINT_DIR))


def load_latest_checkpoint(model_class=ModelClass):
    """Load the most recent checkpoint from persistent storage instead of the base
    model from the Hub, if one exists. Handles both a full base-model save and a
    PEFT/LoRA adapter-only save (the common case once Stage 1 SFT starts -- see
    section 17). Returns (model, processor, tag) or (None, None, None)."""
    ckpts = list_checkpoints()
    if not ckpts:
        print("No checkpoints found yet -- this will be the first one you save.")
        return None, None, None
    latest = ckpts[-1]
    ckpt_dir = os.path.join(CHECKPOINT_DIR, latest)
    print(f"Loading latest checkpoint: {latest}")

    p = AutoProcessor.from_pretrained(ckpt_dir)

    if os.path.exists(os.path.join(ckpt_dir, "adapter_config.json")):
        from peft import PeftModel
        base = model_class.from_pretrained(
            CONFIG["model_name"], quantization_config=bnb_config, device_map="auto",
            cache_dir=os.environ["HF_HUB_CACHE"],
        )
        m = PeftModel.from_pretrained(base, ckpt_dir)
        print("Loaded as a LoRA adapter on top of the base model.")
    else:
        m = model_class.from_pretrained(ckpt_dir, quantization_config=bnb_config, device_map="auto")
        print("Loaded as a full model checkpoint.")

    return m, p, latest


print("Available checkpoints:", list_checkpoints())
# Example, once you actually have something worth saving:
# save_checkpoint(model, processor, tag="baseline-3b-loaded")


## 10. Reproducibility: environment & run log

You're moving between three different machines (Kaggle, Colab, the 4050 laptop, and later
the lab 4090). This appends a small record of each run's environment and package versions
to a persistent log, so results can later be traced back to which machine produced them.


In [ ]:
import platform
import subprocess


def get_pip_freeze(packages):
    try:
        installed = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    except Exception as e:
        return {"error": str(e)}
    lookup = {}
    for line in installed.splitlines():
        if "==" in line:
            name, _, version = line.partition("==")
            lookup[name.lower()] = version
    return {p: lookup.get(p.lower(), "not found") for p in packages}


env_info = {
    "environment": ENV,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "key_packages": get_pip_freeze(
        ["transformers", "accelerate", "peft", "bitsandbytes", "datasets", "huggingface_hub", "qwen-vl-utils"]
    ),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}
print(json.dumps(env_info, indent=2))

run_log_path = os.path.join(PERSIST_DIR, "run_log.jsonl")
with open(run_log_path, "a") as f:
    f.write(json.dumps(env_info) + "\n")
print(f"\nAppended this run's info to {run_log_path}")


## 11. FDI numbering tool

In [ ]:
def fdi_encode(quadrant: int, tooth_position: int) -> int:
    """Combine an FDI quadrant (1-4 permanent, 5-8 deciduous) and a tooth position
    (1-8, counting from the midline) into the two-digit FDI notation DENTEX uses,
    e.g. quadrant=4, tooth_position=8 -> 48 (lower-right third molar)."""
    return quadrant * 10 + tooth_position


def fdi_decode(fdi_number: int):
    """Inverse of fdi_encode: 48 -> (quadrant=4, tooth_position=8)."""
    return divmod(fdi_number, 10)


QUADRANT_NAMES = {
    1: "upper right", 2: "upper left", 3: "lower left", 4: "lower right",
    5: "upper right (deciduous)", 6: "upper left (deciduous)",
    7: "lower left (deciduous)", 8: "lower right (deciduous)",
}


def tool_fdi_label(quadrant: int, tooth_position: int) -> str:
    """The FDI-numbering tool from the proposal (§5.3): turns a (quadrant, tooth
    position) pair -- what a grounding/segmentation model outputs -- into the
    human-readable, clinically standard label the agent reports in its final answer."""
    fdi = fdi_encode(quadrant, tooth_position)
    name = QUADRANT_NAMES.get(quadrant, f"quadrant {quadrant}")
    return f"FDI {fdi} ({name}, tooth position {tooth_position})"


print(tool_fdi_label(4, 8))   # FDI 48: lower right, third molar
print(tool_fdi_label(1, 1))   # FDI 11: upper right, central incisor
print(fdi_decode(48))         # (4, 8)


## 12. Oracle grounding tool (temporary stand-in for Stage 0)

Stage 0 (§5.3/§7 Phase 2) trains a real grounding/segmentation model. Until that exists, this
oracle version reads DENTEX's own ground-truth boxes for a given image, so the agent loop below
can be built and tested end to end right now. It's called the same way a real tool would be, so
swapping in the trained detector later shouldn't require changing anything else in the pipeline.


In [ ]:
def tool_locate_abnormal_teeth(image_id):
    """[ORACLE -- Stage 0 stand-in] Return every annotated abnormal tooth for this
    image: bbox, quadrant, tooth position, FDI label, and diagnosis."""
    rows = annots_df[annots_df["image_id"] == image_id]
    diag_lookup = dict(zip(categories_df["id"], categories_df["name"])) if len(categories_df) else {}
    results = []
    for _, row in rows.iterrows():
        quad = row.get("category_id_1")
        tooth = row.get("category_id_2")
        diag_id = row.get(diag_col) if diag_col else None
        results.append({
            "bbox": list(row["bbox"]),
            "quadrant": quad,
            "tooth_position": tooth,
            "fdi_label": tool_fdi_label(quad, tooth) if pd.notna(quad) and pd.notna(tooth) else None,
            "diagnosis": diag_lookup.get(diag_id),
        })
    return results


example_id = images_df.dropna(subset=["local_path"])["id"].iloc[0]
print(f"Oracle tool output for image_id={example_id}:")
for r in tool_locate_abnormal_teeth(example_id):
    print(" ", r)


## 13. Agent orchestration loop

The core agentic piece (§5.3/§5.4): the model is prompted, generates a reply, that reply is
parsed as either a tool call or a final answer, a tool call gets executed and its result fed
back as a new turn, and this repeats until a final answer or the tool-call budget runs out.

**Expectation to set now**: this runs the *base* model (no SFT/RL yet). Getting a base VLM to
reliably emit well-formed JSON tool calls with no training is unlikely — most rollouts below
will probably hit a parse failure or ignore the schema. That's the expected, normal state of
things pre-fine-tuning, not a bug in this loop; it's precisely the gap Stage 1 SFT closes.


In [ ]:
import re

TOOLS = {}

def register_tool(name, fn, description):
    TOOLS[name] = {"fn": fn, "description": description}

register_tool(
    "zoom_crop",
    lambda image, bbox: tool_zoom_crop(image, bbox),
    'zoom_crop(bbox=[x,y,w,h]): return a zoomed-in crop around a region of the CURRENT image.',
)
register_tool(
    "enhance_contrast",
    lambda image, factor=1.5: tool_enhance_contrast(image, factor),
    "enhance_contrast(factor=1.5): return a contrast-enhanced version of the CURRENT image.",
)
register_tool(
    "locate_abnormal_teeth",
    lambda image_id: tool_locate_abnormal_teeth(image_id),
    "locate_abnormal_teeth(): [ORACLE, Stage-0 stand-in] list all abnormal teeth already "
    "known in this image, each with bbox, FDI label, and diagnosis.",
)

AGENT_SYSTEM_PROMPT = (
    "You are a dental radiograph analysis agent. You can call tools to gather evidence "
    'before answering. To call a tool, respond with EXACTLY one JSON object on its own '
    'line: {"tool": "<name>", "args": {...}}. When ready to give your final answer, respond '
    'with EXACTLY one JSON object: {"final_answer": {"quadrant": <1-4>, "tooth_position": '
    '<1-8>, "diagnosis": "<caries|deep_caries|periapical_lesion|impacted_tooth>", '
    '"confidence": <0-1>}}. Do not include any other text outside the JSON object.\n\n'
    "Available tools:\n"
    + "\n".join(f"- {v['description']}" for v in TOOLS.values())
)
print(AGENT_SYSTEM_PROMPT)


In [ ]:
def parse_agent_json(text):
    """Extract the first JSON object from the model's output. Returns None if
    nothing parseable was found -- a format failure the R_format reward penalizes."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def generate_agent_reply(messages, max_new_tokens=200, return_ids=False):
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    prompt_len = inputs.input_ids.shape[1]
    trimmed = generated[0][prompt_len:]
    text = processor.decode(trimmed, skip_special_tokens=True)
    if return_ids:
        return text, prompt_len, trimmed.detach().cpu()
    return text


In [ ]:
def run_agent(image_id, max_tool_calls=4, verbose=True):
    """The core agentic loop. Returns a trajectory dict the reward functions in the
    next section can score directly, plus `assistant_token_spans` (prompt_len, token_ids)
    per assistant turn -- used later for GRPO's token-level credit assignment (section 18).
    The outer `for` loop's range is the actual safety bound on total steps, regardless of
    the budget-exhausted branch below."""
    row = images_df[images_df["id"] == image_id].iloc[0]
    current_image = Image.open(row["local_path"]).convert("RGB")

    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": current_image},
            {"type": "text", "text": f"Analyze this panoramic X-ray (image_id={image_id})."},
        ]},
    ]

    trajectory = {
        "image_id": image_id, "turns": [], "tool_calls": 0, "final_answer": None,
        "assistant_token_spans": [],
    }

    for _ in range(max_tool_calls + 1):
        reply, prompt_len, gen_ids = generate_agent_reply(messages, return_ids=True)
        trajectory["assistant_token_spans"].append({"prompt_len": prompt_len, "token_ids": gen_ids})
        parsed = parse_agent_json(reply)
        trajectory["turns"].append({"raw_output": reply, "parsed": parsed})

        if parsed is None:
            trajectory["format_ok"] = False
            break

        if "final_answer" in parsed:
            trajectory["final_answer"] = parsed["final_answer"]
            trajectory["format_ok"] = True
            break

        if "tool" in parsed and parsed["tool"] in TOOLS:
            if trajectory["tool_calls"] >= max_tool_calls:
                messages.append({"role": "assistant", "content": reply})
                messages.append({"role": "user", "content": (
                    "Tool-call budget exhausted. Give your final_answer now."
                )})
                continue

            trajectory["tool_calls"] += 1
            tool_name = parsed["tool"]
            tool_args = parsed.get("args", {}) or {}
            tool_fn = TOOLS[tool_name]["fn"]
            try:
                if tool_name == "locate_abnormal_teeth":
                    tool_result = tool_fn(image_id)
                elif tool_name == "zoom_crop":
                    tool_result = tool_fn(current_image, tool_args.get("bbox"))
                    current_image = tool_result
                elif tool_name == "enhance_contrast":
                    tool_result = tool_fn(current_image, tool_args.get("factor", 1.5))
                    current_image = tool_result
                else:
                    tool_result = tool_fn(**tool_args)
                tool_ok = True
            except Exception as e:
                tool_result = f"Tool error: {e}"
                tool_ok = False

            trajectory["turns"][-1]["tool_ok"] = tool_ok
            messages.append({"role": "assistant", "content": reply})

            if isinstance(tool_result, Image.Image):
                messages.append({"role": "user", "content": [
                    {"type": "image", "image": tool_result},
                    {"type": "text", "text": f"Tool '{tool_name}' result (new image above)."},
                ]})
            else:
                messages.append({"role": "user", "content": [
                    {"type": "text", "text": f"Tool '{tool_name}' result: {tool_result}"},
                ]})
        else:
            trajectory["format_ok"] = False
            break

    trajectory["messages"] = messages  # kept for GRPO's full-trajectory re-tokenization (section 18)
    if verbose:
        print(f"Trajectory for image_id={image_id}: {trajectory['tool_calls']} tool call(s), "
              f"final_answer={trajectory['final_answer']}")
    return trajectory


demo_trajectory = run_agent(example_id)


## 14. Reward functions (§5.5)

In [ ]:
def reward_format(trajectory):
    """R_format: did the agent stay inside the required JSON schema at every turn?"""
    return 1.0 if trajectory.get("format_ok") else 0.0


def reward_tool_validity(trajectory):
    """R_tool_validity: fraction of executed tool calls that ran successfully."""
    tool_turns = [t for t in trajectory["turns"] if (t.get("parsed") or {}).get("tool")]
    if not tool_turns:
        return 0.0
    ok = [t for t in tool_turns if t.get("tool_ok")]
    return len(ok) / len(tool_turns)


def reward_efficiency(trajectory, budget=4, penalty_per_extra=0.1):
    """R_efficiency: penalize tool calls beyond the budget."""
    extra = max(0, trajectory["tool_calls"] - budget)
    return -extra * penalty_per_extra


def reward_accuracy(trajectory, ground_truth, full_credit=1.0, quad_tooth_credit=0.5, quad_credit=0.25):
    """R_accuracy: graded credit -- full credit for quadrant + tooth position + diagnosis
    all correct, partial credit otherwise, matching DENTEX's hierarchical annotations."""
    ans = trajectory.get("final_answer")
    if not ans:
        return 0.0
    quad_ok = ans.get("quadrant") == ground_truth.get("quadrant")
    tooth_ok = ans.get("tooth_position") == ground_truth.get("tooth_position")
    diag_ok = str(ans.get("diagnosis", "")).lower() == str(ground_truth.get("diagnosis", "")).lower()

    if quad_ok and tooth_ok and diag_ok:
        return full_credit
    if quad_ok and tooth_ok:
        return quad_tooth_credit
    if quad_ok:
        return quad_credit
    return 0.0


def combine_reward(trajectory, ground_truth, weights=None):
    """R = w_acc*R_accuracy + w_fmt*R_format + w_tool*R_tool_validity + w_eff*R_efficiency
    (§5.5). R_judge is intentionally excluded here -- it needs an external LLM-judge API
    call (section 16), not something computable offline."""
    weights = weights or {"acc": 1.0, "fmt": 0.2, "tool": 0.2, "eff": 0.1}
    components = {
        "accuracy": reward_accuracy(trajectory, ground_truth),
        "format": reward_format(trajectory),
        "tool_validity": reward_tool_validity(trajectory),
        "efficiency": reward_efficiency(trajectory),
    }
    total = (
        weights["acc"] * components["accuracy"]
        + weights["fmt"] * components["format"]
        + weights["tool"] * components["tool_validity"]
        + weights["eff"] * components["efficiency"]
    )
    return total, components


## 15. End-to-end demo: agent loop + reward, together

In [ ]:
gt_rows = tool_locate_abnormal_teeth(example_id)
if gt_rows:
    ground_truth = {
        "quadrant": gt_rows[0]["quadrant"],
        "tooth_position": gt_rows[0]["tooth_position"],
        "diagnosis": gt_rows[0]["diagnosis"],
    }
    total_reward, components = combine_reward(demo_trajectory, ground_truth)
    print("Ground truth:      ", ground_truth)
    print("Agent final answer:", demo_trajectory["final_answer"])
    print("Reward components: ", components)
    print("Total reward:      ", round(total_reward, 3))
    print(
        "\nIf format/accuracy are near zero, that's expected: this is the BASE model with "
        "no fine-tuning yet (see the note at the top of section 13). Stage 1 SFT exists "
        "specifically to teach reliable schema adherence before RL (Stage 2) ever sees it."
    )
else:
    print(f"No annotated abnormalities found for image_id={example_id} -- pick a "
          f"different id from images_df and re-run section 13's demo.")


## 16. Trace generation + verification pipeline (Aim 1, §5.2)

Needs **no GPU** — pure API calls — so this is genuinely runnable right now, independent of
everything else in this notebook, as long as you have API access. Uses two *different* model
families for generation vs. verification (§5.2 step 4: bias control, so the verifier isn't
just grading its own family's mistakes) — generation is set up below for Gemini free-tier
across however many keys you have; verification stays a different family (Anthropic by
default). Fill in keys before running:
```python
os.environ["GEMINI_API_KEYS"] = "key_one,key_two,key_three"  # however many you actually have
os.environ["ANTHROPIC_API_KEY"] = "..."                      # verifier -- a different family
```


In [ ]:
import datetime


class AllKeysExhaustedToday(Exception):
    """Raised by GeminiKeyPool when every configured key has hit today's request cap.
    run_aim1_batch (further down) catches this specifically and stops cleanly rather than
    retrying with backoff -- a daily quota won't reset in seconds, only overnight."""
    pass


GEMINI_API_KEYS = [k.strip() for k in os.environ.get("GEMINI_API_KEYS", "").split(",") if k.strip()]
# One comma-separated env var, however many keys you actually have (3, 7, 9, ...) -- nothing
# here assumes a fixed count; len(GEMINI_API_KEYS) is computed from whatever you set above.
#
# IMPORTANT: this only multiplies your effective daily throughput if each key belongs to a
# SEPARATE Google Cloud project -- Gemini's free-tier limits are enforced per PROJECT, not
# per key, so several keys under the same project share one quota pool and this whole scheme
# gains nothing. Check each key's owning project in AI Studio (aistudio.google.com) first.

GEMINI_MODEL = "gemini-3.6-flash"  # <- adjust to whatever exact model ID your account exposes

# Conservative placeholders, NOT verified numbers for this specific model -- AI Studio shows
# YOUR project's live current limits; check there and adjust these two before a real run.
GEMINI_RPM_LIMIT = 5
GEMINI_RPD_LIMIT = 20
GEMINI_SAFETY_MARGIN = 1  # target 95% of the stated limit, not 100%, to absorb timing jitter

GEMINI_KEY_STATE_PATH = os.path.join(DATA_DIR, "gemini_key_state.json")


class GeminiKeyPool:
    """Round-robins calls across GEMINI_API_KEYS, respecting each key's own RPM and RPD
    limits, persisting daily-usage state to disk under DATA_DIR so it survives a session
    restart -- including into a new calendar day, at which point each key's counter resets
    automatically (detected by comparing today's date to the saved state, not anything you
    reset by hand). Keys are identified by their POSITION in the list, never by their value,
    so this state file is safe to sync alongside your other artifacts without leaking anything.
    """

    def __init__(self, keys, rpm=GEMINI_RPM_LIMIT, rpd=GEMINI_RPD_LIMIT,
                 safety_margin=GEMINI_SAFETY_MARGIN, state_path=GEMINI_KEY_STATE_PATH):
        self.keys = keys
        self.rpm_limit = max(1, int(rpm * safety_margin))
        self.rpd_limit = max(1, int(rpd * safety_margin))
        self.state_path = state_path
        self.state = self._load()
        self._next_idx = 0
        self._reset_if_new_day()

    def _load(self):
        if os.path.exists(self.state_path):
            with open(self.state_path) as f:
                return json.load(f)
        return {}

    def _save(self):
        with open(self.state_path, "w") as f:
            json.dump(self.state, f, indent=2)

    def _reset_if_new_day(self):
        today = datetime.date.today().isoformat()
        changed = False
        for i in range(len(self.keys)):
            kid = str(i)
            entry = self.state.get(kid)
            if entry is None or entry.get("date") != today:
                self.state[kid] = {"date": today, "calls_today": 0, "last_call_ts": 0.0}
                changed = True
        if changed:
            self._save()

    def acquire(self):
        """Blocks only as long as needed for the chosen key's own RPM spacing, then
        returns (index, key). Raises AllKeysExhaustedToday if every key has hit its daily
        cap -- callers should stop (not retry-with-backoff) and try again after the reset."""
        if not self.keys:
            raise ValueError("GEMINI_API_KEYS is empty -- set it before calling provider='gemini'.")
        self._reset_if_new_day()

        for _ in range(len(self.keys)):
            idx = self._next_idx % len(self.keys)
            self._next_idx += 1
            entry = self.state[str(idx)]
            if entry["calls_today"] >= self.rpd_limit:
                continue  # this key is done for today -- try the next one in the rotation
            elapsed = time.time() - entry["last_call_ts"]
            min_interval = 60.0 / self.rpm_limit
            if elapsed < min_interval:
                time.sleep(min_interval - elapsed)
            entry["last_call_ts"] = time.time()
            entry["calls_today"] += 1
            self._save()
            return idx, self.keys[idx]

        raise AllKeysExhaustedToday(
            f"All {len(self.keys)} Gemini key(s) have hit today's cap "
            f"({self.rpd_limit} calls/key after the safety margin). Gemini's free tier "
            f"resets around midnight Pacific time -- re-run this same cell tomorrow; "
            f"resume=True in run_aim1_batch will pick up exactly where this left off."
        )

    def status(self):
        """How much of today's budget is left per key -- check before a big batch, or
        after a run stops early."""
        self._reset_if_new_day()
        return pd.DataFrame([{
            "key_index": i,
            "calls_today": self.state[str(i)]["calls_today"],
            "rpd_limit": self.rpd_limit,
            "remaining_today": max(0, self.rpd_limit - self.state[str(i)]["calls_today"]),
        } for i in range(len(self.keys))])


gemini_pool = GeminiKeyPool(GEMINI_API_KEYS)
if GEMINI_API_KEYS:
    print(f"Gemini key pool ready: {len(GEMINI_API_KEYS)} key(s) detected.")
    print(gemini_pool.status())
else:
    print("GEMINI_API_KEYS is empty -- set it (see the comment above) before using "
          "provider='gemini' anywhere below.")


In [ ]:
GENERATOR_PROVIDER = "gemini"        # multi-key free-tier pool, rate-limited above
VERIFIER_PROVIDER = "anthropic"      # deliberately a different family from the generator
GENERATOR_MODEL = GEMINI_MODEL
VERIFIER_MODEL = "claude-opus-4-1"   # placeholder -- check current model names before running


def call_llm(provider, model, system_prompt, user_content, image=None, max_tokens=600):
    """Minimal provider-agnostic chat call. Add a branch for other providers as needed."""
    import base64
    from io import BytesIO

    def image_to_b64(img):
        buf = BytesIO()
        img.convert("RGB").save(buf, format="JPEG", quality=90)
        return base64.b64encode(buf.getvalue()).decode()

    if provider == "openai":
        from openai import OpenAI
        client = OpenAI()
        content = [{"type": "text", "text": user_content}]
        if image is not None:
            content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_to_b64(image)}"},
            })
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": content},
            ],
            max_tokens=max_tokens,
        )
        return resp.choices[0].message.content

    elif provider == "anthropic":
        import anthropic
        client = anthropic.Anthropic()
        content = [{"type": "text", "text": user_content}]
        if image is not None:
            content.insert(0, {
                "type": "image",
                "source": {"type": "base64", "media_type": "image/jpeg", "data": image_to_b64(image)},
            })
        resp = client.messages.create(
            model=model, max_tokens=max_tokens, system=system_prompt,
            messages=[{"role": "user", "content": content}],
        )
        return resp.content[0].text

    elif provider == "gemini":
        # Uses the current unified SDK (google-genai) -- NOT the deprecated
        # google-generativeai package. This SDK's exact call surface has moved before and
        # may move again; if this errors, check ai.google.dev/gemini-api/docs for the
        # current pattern and adjust just this branch -- nothing else here depends on it.
        # Goes through gemini_pool.acquire() FIRST so the RPM/RPD limits (section above)
        # are respected before every single call, not just checked in bulk beforehand.
        from google import genai
        from google.genai import types

        _, api_key = gemini_pool.acquire()
        client = genai.Client(api_key=api_key)

        parts = [user_content]
        if image is not None:
            parts.append(image.convert("RGB"))  # google-genai accepts PIL Images directly

        resp = client.models.generate_content(
            model=model, contents=parts,
            config=types.GenerateContentConfig(system_instruction=system_prompt, max_output_tokens=max_tokens),
        )
        return resp.text

    else:
        raise NotImplementedError(f"Add a branch for provider={provider!r}.")


In [ ]:
GENERATOR_SYSTEM_PROMPT = (
    "You are generating TRAINING DATA for a dental radiograph analysis agent. Given an "
    "X-ray image and the KNOWN correct answer, write a plausible step-by-step reasoning "
    "trace a model could have produced to reach that answer, including where it would "
    "zoom in and what visual evidence it would cite. Include one zoom_crop tool call "
    "(with an approximate bbox around the finding) before the final answer. Follow this "
    "schema across multiple lines:\n"
    "<think>...</think>\n"
    '{"tool": "zoom_crop", "args": {"bbox": [x, y, w, h]}}\n'
    "<think>...</think>\n"
    '{"final_answer": {"quadrant": ..., "tooth_position": ..., "diagnosis": "...", "confidence": ...}}'
)

VERIFIER_SYSTEM_PROMPT = (
    "You are a strict verifier, not a rewriter. Given an X-ray image, the KNOWN correct "
    "answer, and a candidate reasoning trace, judge ONLY whether every claim in the trace "
    "is actually supported by the image and the known answer -- reject any trace asserting "
    "something the image/label does not support, even if the final answer happens to be "
    'correct. Respond with exactly one JSON object: {"grounded": true/false, "reason": "..."}.'
)


def generate_trace(image, ground_truth, k=3):
    """Generate k candidate traces for one example (self-consistency, §5.2 step 3)."""
    user_content = f"Known correct answer: {json.dumps(ground_truth)}"
    return [
        call_llm(GENERATOR_PROVIDER, GENERATOR_MODEL, GENERATOR_SYSTEM_PROMPT, user_content, image=image)
        for _ in range(k)
    ]


def verify_trace(image, ground_truth, trace):
    """Verify one trace with a DIFFERENT model family than the generator (bias control)."""
    user_content = f"Known correct answer: {json.dumps(ground_truth)}\n\nCandidate trace:\n{trace}"
    raw = call_llm(VERIFIER_PROVIDER, VERIFIER_MODEL, VERIFIER_SYSTEM_PROMPT, user_content, image=image)
    parsed = parse_agent_json(raw)
    return parsed if parsed else {"grounded": False, "reason": "verifier output unparseable"}


def build_trace_example(image_id, k=3):
    """Full Aim 1 pipeline for one image: generate k candidates, verify each, keep only
    grounded traces."""
    row = images_df[images_df["id"] == image_id].iloc[0]
    image = Image.open(row["local_path"]).convert("RGB")
    gt_rows = tool_locate_abnormal_teeth(image_id)
    if not gt_rows:
        return None
    ground_truth = {
        "quadrant": gt_rows[0]["quadrant"],
        "tooth_position": gt_rows[0]["tooth_position"],
        "diagnosis": gt_rows[0]["diagnosis"],
    }

    candidates = generate_trace(image, ground_truth, k=k)
    verified = [t for t in candidates if verify_trace(image, ground_truth, t).get("grounded")]

    return {
        "image_id": image_id,
        "ground_truth": ground_truth,
        "n_candidates": len(candidates),
        "n_verified": len(verified),
        "verified_traces": verified,
    }


In [ ]:
def to_jsonable(obj):
    """Recursively convert numpy scalar/array types to native Python types. Values read
    from a pandas DataFrame (e.g. a quadrant id) are often numpy.int64, which json.dump
    can't serialize -- and json.dump(..., default=str) would 'fix' that by turning them
    into strings, silently breaking later equality checks like `ans['quadrant'] ==
    ground_truth['quadrant']` (3 == '3' is False). This preserves the actual type instead."""
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


In [ ]:
# Pilot run -- capped to a couple of examples so you can validate the pipeline design before
# spending real API budget on the full ~700-case training set (§5.2, §7 Phase 1). Requires the
# API keys above; skipped automatically otherwise so this cell is always safe to run.
PILOT_N = 2

if os.environ.get("OPENAI_API_KEY") and os.environ.get("ANTHROPIC_API_KEY"):
    _trainable_pool = images_df[~images_df["id"].isin(holdout_ids)].dropna(subset=["local_path"])
    pilot_image_ids = _trainable_pool["id"].sample(PILOT_N, random_state=CONFIG["seed"]).tolist()
    pilot_results = []
    for iid in pilot_image_ids:
        result = build_trace_example(iid, k=3)
        if result:
            pilot_results.append(result)
            print(f"image_id={iid}: {result['n_verified']}/{result['n_candidates']} traces passed verification")

    trace_cache_path = os.path.join(DATA_DIR, "pilot_traces.json")
    with open(trace_cache_path, "w") as f:
        json.dump(to_jsonable(pilot_results), f, indent=2)
    print(f"\nSaved pilot traces to {trace_cache_path}")
else:
    print("Skipping the pilot run -- set OPENAI_API_KEY and ANTHROPIC_API_KEY (or adapt "
          "call_llm for your providers) to actually generate and verify traces.")


### Scaling up beyond the pilot: rate-limit-aware batching

The pilot run above is intentionally tiny. Scaling to the full ~700-image training set needs
two things a 2-example loop doesn't: resilience to transient API failures (rate limits,
timeouts), and a rough cost estimate *before* committing budget, not after.


In [ ]:
def estimate_aim1_api_calls(n_images, k=3):
    """Rough call-count estimate before spending real budget: k generator calls + k
    verifier calls per image (one verification per candidate trace). Also projects how
    many DAYS the generator side will take given the Gemini pool's actual configured
    daily budget -- the number that actually matters for planning around "resume
    tomorrow", not just a raw call count."""
    calls_per_image = 2 * k
    total = n_images * calls_per_image
    generator_calls = n_images * k
    print(f"{n_images} images x {calls_per_image} calls/image (k={k} generate + k verify) "
          f"= ~{total} total API calls (~{generator_calls} generator, ~{generator_calls} verifier).")

    if GEMINI_API_KEYS:
        daily_budget = len(GEMINI_API_KEYS) * gemini_pool.rpd_limit
        days = -(-generator_calls // daily_budget)  # ceiling division
        print(f"\nGenerator (Gemini): {len(GEMINI_API_KEYS)} key(s) x {gemini_pool.rpd_limit} "
              f"calls/day (after the safety margin) = {daily_budget} calls/day budget.")
        print(f"At that rate, {generator_calls} generator calls take ~{days} day(s), "
              f"assuming you re-run run_aim1_batch() once per day as each daily cap resets.")
    else:
        print("\nGEMINI_API_KEYS isn't set yet -- can't project days-to-complete until it is.")
    return total


def run_aim1_batch(image_ids, k=3, cache_path=None, resume=True, max_retries=3, retry_delay=5.0):
    """Scale-up of the pilot run: generate + verify traces across many images, with
    incremental disk caching (resume=True survives a died session -- including into a new
    calendar day) and exponential-backoff retry for ordinary transient failures. A daily
    quota running out (AllKeysExhaustedToday, from the Gemini pool above) is handled
    separately below -- retrying that with backoff would just waste time, since it won't
    resolve in seconds."""
    results, done_ids = [], set()
    if cache_path and resume and os.path.exists(cache_path):
        with open(cache_path) as f:
            results = json.load(f)
        done_ids = {r["image_id"] for r in results}
        print(f"Resuming: {len(done_ids)} image(s) already processed in {cache_path}")

    todo = [i for i in image_ids if i not in done_ids]
    total_candidates = total_verified = 0
    for idx, image_id in enumerate(todo):
        result = None
        try:
            for attempt in range(max_retries):
                try:
                    result = build_trace_example(image_id, k=k)
                    break
                except AllKeysExhaustedToday:
                    raise  # bubble straight out -- don't retry-with-backoff a daily quota
                except Exception as e:
                    wait = retry_delay * (2 ** attempt)
                    print(f"  image_id={image_id}: attempt {attempt + 1}/{max_retries} failed "
                          f"({e}); retrying in {wait:.0f}s")
                    time.sleep(wait)
            else:
                print(f"  image_id={image_id}: giving up after {max_retries} attempts, skipping")
        except AllKeysExhaustedToday as e:
            if cache_path:
                with open(cache_path, "w") as f:
                    json.dump(results, f, indent=2)
            print(f"\n{e}")
            print(f"Stopped after {idx}/{len(todo)} image(s) from this run "
                  f"({len(results)} total processed so far, saved to {cache_path}).")
            print("This is the expected 'come back tomorrow' stop, not a crash -- re-run "
                  "this exact call (same cache_path, resume=True) after the next Gemini "
                  "daily reset and it continues from here, not from the start.")
            return results

        if result:
            results.append(to_jsonable(result))
            total_candidates += result["n_candidates"]
            total_verified += result["n_verified"]

        if cache_path:
            with open(cache_path, "w") as f:
                json.dump(results, f, indent=2)

        if (idx + 1) % 10 == 0 or idx == len(todo) - 1:
            rate = total_verified / max(total_candidates, 1)
            print(f"  {idx + 1}/{len(todo)} done -- verified rate so far: {rate:.1%}")

    print(f"\nAll {len(todo)} image(s) in this run processed -- nothing left to resume.")
    return results


print("Ready for the full-scale run once GEMINI_API_KEYS and ANTHROPIC_API_KEY are set, e.g.:\n"
      "  pool_ids = images_df[~images_df['id'].isin(holdout_ids)].dropna(subset=['local_path'])\n"
      "  estimate_aim1_api_calls(len(pool_ids), k=3)\n"
      "  print(gemini_pool.status())  # check today's remaining budget before a long run\n"
      "  full_results = run_aim1_batch(pool_ids['id'].tolist(),\n"
      "                                 cache_path=os.path.join(DATA_DIR, 'full_traces.json'))\n"
      "  # if it stops early with an AllKeysExhaustedToday message, just re-run the same\n"
      "  # call tomorrow -- resume=True (the default) picks up exactly where it left off")


## 17. Stage 1: SFT on the verified traces

LoRA/QLoRA fine-tuning (§5.4 Stage 1) to teach the base model reliable schema adherence --
the exact gap section 13's honest-expectation note flagged. A manual training loop is used
here rather than a specific trainer class, so it doesn't depend on an exact library API
surface that may have moved on by the time you run this; swap in TRL's SFTTrainer later if
you prefer, once you've confirmed its current API against your installed version.


In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel

# Resume from a previous SFT/GRPO checkpoint if one exists, instead of double-wrapping
# a fresh LoRA adapter onto an already-adapted model.
resumed_model, resumed_processor, resumed_tag = load_latest_checkpoint()
if resumed_model is not None:
    model = resumed_model
    processor = resumed_processor
    print(f"Resuming training from checkpoint: {resumed_tag}")
else:
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        # Standard Qwen2-family attention + MLP projection module names. If this errors
        # with "target modules not found", print(model) and adjust these names to match.
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)
    print("Starting LoRA fine-tuning from the base model (no prior checkpoint found).")

model.print_trainable_parameters()


In [ ]:
def build_sft_example(image, prompt_text_content, target_trace_text):
    """Build one training example: prompt tokens (system + user image/text) are
    masked out of the loss (label = -100); only the target trace's tokens contribute,
    which is the standard SFT masking approach for causal-LM fine-tuning."""
    prompt_messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt_text_content},
        ]},
    ]
    prompt_text = processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + target_trace_text + processor.tokenizer.eos_token

    image_inputs, video_inputs = process_vision_info(prompt_messages)
    prompt_enc = processor(text=[prompt_text], images=image_inputs, videos=video_inputs, return_tensors="pt")
    full_enc = processor(text=[full_text], images=image_inputs, videos=video_inputs, return_tensors="pt")

    labels = full_enc["input_ids"].clone()
    prompt_len = prompt_enc["input_ids"].shape[1]
    labels[:, :prompt_len] = -100
    full_enc["labels"] = labels
    return full_enc


class TraceSFTDataset(torch.utils.data.Dataset):
    """Expands to one training example per verified trace (not per image) -- an
    image with 3 verified traces contributes 3 training examples."""

    def __init__(self, trace_examples):
        self.samples = []
        for ex in trace_examples:
            matches = images_df[images_df["id"] == ex["image_id"]]
            if matches.empty:
                continue
            row = matches.iloc[0]
            for trace_text in ex.get("verified_traces", []):
                self.samples.append({
                    "image_path": row["local_path"],
                    "prompt_text": f"Analyze this panoramic X-ray (image_id={ex['image_id']}).",
                    "target_text": trace_text,
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        image = Image.open(s["image_path"]).convert("RGB")
        return build_sft_example(image, s["prompt_text"], s["target_text"])


In [ ]:
def load_trace_dataset(path=None):
    """Load whatever verified traces exist on disk -- the section-16 pilot by default,
    or a larger trace file you've generated separately (pass its path explicitly)."""
    path = path or os.path.join(DATA_DIR, "pilot_traces.json")
    if not os.path.exists(path):
        print(f"No trace file found at {path} -- run section 16 (with API keys set) "
              f"first, or point this at a larger trace file you've generated separately.")
        return []
    with open(path) as f:
        return json.load(f)


trace_examples = load_trace_dataset()
sft_dataset = TraceSFTDataset(trace_examples) if trace_examples else None
print(f"SFT training examples available: {len(sft_dataset) if sft_dataset else 0}")


In [ ]:
def sft_train(model, dataset, epochs=1, lr=1e-4, log_every=1):
    """Minimal manual SFT loop, batch size 1 (fine for free-tier 16GB GPUs; increase
    batch size / add gradient accumulation once you're on the 4090)."""
    if not dataset or len(dataset) == 0:
        print("No training data available yet -- nothing to train on.")
        return model

    model.train()
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)

    step = 0
    total_steps = len(dataset) * epochs
    for epoch in range(epochs):
        for i in range(len(dataset)):
            batch = dataset[i]
            batch = {k: v.to(model.device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            step += 1
            if step % log_every == 0:
                print(f"epoch {epoch} step {step}/{total_steps}  loss={loss.item():.4f}")

    model.eval()
    return model


if sft_dataset:
    model = sft_train(model, sft_dataset, epochs=1)
    save_checkpoint(model, processor, tag="sft-3b-v1")
else:
    print("Skipping training -- generate trace data in section 16 first (real API keys, "
          "a larger PILOT_N or a dedicated batch run), then re-run this cell.")


## 18. Stage 2: GRPO, now with a KL penalty and PPO-style clipping

**Read this before running it.** This is a from-scratch reference implementation of GRPO's
core mechanics (group rollouts → graded reward → group-normalized advantage → policy-gradient
update), adapted to multi-turn tool-calling. It now includes two of the three things the
previous version of this section flagged as missing:

- **A KL penalty against a frozen reference policy** — at essentially no extra memory cost.
  The reference policy is just this same model with its LoRA adapter temporarily disabled
  (`model.disable_adapter()`), since disabling the adapter recovers the original base model.
  No second copy of the model needs to live in memory, which matters a lot on 16GB free-tier
  GPUs.
- **PPO-style ratio clipping** — meaningful only once `epochs_per_batch > 1`: with a single
  epoch (the previous default), the "old" and "new" policy are the same weights, the ratio is
  ~1, and clipping is a no-op — which is exactly why the earlier version omitted it rather
  than include dead code. Set `epochs_per_batch=2+` to take multiple gradient steps on the
  same collected group of rollouts, at `epochs_per_batch` times the compute per call.

Still deliberately simplified, and still a way to understand and sanity-check the mechanics
at small scale (§7 Phase 2's "smoke test"), not a production trainer — move to TRL/EasyR1
(§5.4) once validated:
- The policy-gradient/clip/KL terms operate on a single aggregate advantage per rollout
  (standard GRPO), but the ratio and KL are computed per-token and mask-averaged, not
  collapsed to one sequence-level number, which is the more standard formulation.
- No multi-epoch mini-batching across *different* prompts within an epoch — `epochs_per_batch`
  repeats over the same small batch, it doesn't shuffle in new data.

**One assumption worth knowing**: computing the loss requires knowing exactly which tokens
in the final trajectory the model actually generated (vs. tool-result/user text that was
injected). This re-tokenizes the full finished conversation and locates each assistant turn's
span using the prompt length recorded during rollout collection. That alignment assumes the
tokenizer produces the same token boundaries whether text is tokenized incrementally (as
during generation) or as part of a longer final string (generally true for BPE tokenizers,
but not something this notebook has verified against your specific installed version) --
the validation cell right after the loss function lets you check this yourself before
trusting it for real training.


In [ ]:
def compute_group_advantages(rewards):
    """A_i = (r_i - mean(r)) / (std(r) + eps) -- GRPO's group-relative advantage,
    computed once per group of rollouts sampled for the SAME prompt/image."""
    rewards = torch.tensor(rewards, dtype=torch.float32)
    mean, std = rewards.mean(), rewards.std(unbiased=False)
    return (rewards - mean) / (std + 1e-4)


In [ ]:
import contextlib


def compute_token_log_probs(enc, use_reference=False):
    """Per-token log-probs + loss mask, for either the current policy (default) or the
    frozen reference policy (use_reference=True). The reference policy is obtained by
    temporarily disabling the LoRA adapter -- the base model underneath IS the reference
    policy, so this needs no second copy of the model in memory. Falls back to the current
    policy (no true reference) if `model` isn't LoRA-wrapped, since there's nothing else
    to disable."""
    labels = enc["labels"]
    model_inputs = {k: v for k, v in enc.items() if k != "labels"}

    use_disable = use_reference and hasattr(model, "disable_adapter")
    context = model.disable_adapter() if use_disable else contextlib.nullcontext()
    with context:
        with torch.set_grad_enabled(not use_reference):
            outputs = model(**model_inputs)

    logits = outputs.logits[:, :-1, :]
    shift_labels = labels[:, 1:].to(logits.device)
    log_probs = torch.log_softmax(logits, dim=-1)
    token_log_probs = torch.gather(log_probs, 2, shift_labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    mask = (shift_labels != -100).float()
    return token_log_probs, mask


In [ ]:
def collect_grpo_group(image_id, ground_truth, group_size=4, max_tool_calls=4):
    """Sample `group_size` rollouts for one image (§5.4: default group_size=4 on
    free-tier GPUs, 8 once you have 4090 time -- see the compute-budget estimate in
    section 7). Also captures each trajectory's OLD (pre-update) per-token log-probs
    under no_grad, right after generation -- what a PPO-style ratio is computed against
    in later epochs over this same group."""
    trajectories, rewards, old_log_probs_list, masks_list = [], [], [], []
    model.eval()
    for _ in range(group_size):
        traj = run_agent(image_id, max_tool_calls=max_tool_calls, verbose=False)
        total_reward, _ = combine_reward(traj, ground_truth)
        trajectories.append(traj)
        rewards.append(total_reward)

        enc = build_full_trajectory_labels(traj)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            old_lp, mask = compute_token_log_probs(enc, use_reference=False)
        old_log_probs_list.append(old_lp.detach())
        masks_list.append(mask)

    return trajectories, rewards, old_log_probs_list, masks_list


In [ ]:
def build_full_trajectory_labels(trajectory):
    """Re-tokenize the finished conversation and unmask only the spans this policy
    actually generated (one per assistant turn), using each turn's recorded prompt_len
    as the split point. Returns encoded inputs + labels ready for a forward pass."""
    messages = trajectory["messages"]
    full_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    image_inputs, video_inputs = process_vision_info(messages)
    full_enc = processor(text=[full_text], images=image_inputs, videos=video_inputs, return_tensors="pt")

    labels = torch.full_like(full_enc["input_ids"], -100)
    for span in trajectory["assistant_token_spans"]:
        start = span["prompt_len"]
        gen_ids = span["token_ids"]
        end = start + len(gen_ids)
        if end <= labels.shape[1]:
            labels[0, start:end] = gen_ids
    full_enc["labels"] = labels
    return full_enc


def validate_span_alignment(trajectory, verbose=True):
    """Sanity check for the assumption above: decode the unmasked (non--100) label
    tokens and compare them against what the assistant turns actually said. Run this
    on a few trajectories before trusting the GRPO loss below."""
    enc = build_full_trajectory_labels(trajectory)
    labels = enc["labels"][0]
    unmasked_ids = labels[labels != -100]
    decoded = processor.tokenizer.decode(unmasked_ids, skip_special_tokens=True)
    actual = " ".join(t["raw_output"] for t in trajectory["turns"])
    if verbose:
        print("Decoded from labels: ", decoded[:300])
        print("Actual turn outputs: ", actual[:300])
    return decoded.strip() == actual.strip()


print("Span alignment matches exactly:", validate_span_alignment(demo_trajectory))
print(
    "If this prints False, inspect the two printed strings above -- a partial/fuzzy "
    "mismatch (extra whitespace, a stray special token) is usually fixable in "
    "build_full_trajectory_labels; a totally unrelated mismatch means the span "
    "tracking itself is broken and needs debugging before you trust the loss below."
)


In [ ]:
def grpo_step(image_ids_and_gts, group_size=4, max_tool_calls=4, lr=1e-5,
              epochs_per_batch=1, clip_eps=0.2, kl_beta=0.04):
    """One GRPO update cycle. Rollouts are collected ONCE from the current policy (GRPO's
    on-policy design); `epochs_per_batch` then controls how many gradient passes are taken
    over that same fixed batch, each scored against the group-normalized advantage, a
    PPO-style clipped ratio relative to the pre-update log-probs, and a KL penalty toward
    the frozen (adapter-disabled) reference policy."""
    if not hasattr(model, "disable_adapter") and kl_beta > 0:
        print("model has no disable_adapter() (not LoRA-wrapped) -- forcing kl_beta=0, "
              "since there's no reference policy to disable into.")
        kl_beta = 0.0

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)

    # Collect once, per GRPO's on-policy design -- rollouts come from the policy as it is
    # right now, before any of this call's updates.
    groups, all_rewards = [], []
    for image_id, ground_truth in image_ids_and_gts:
        trajs, rewards, old_lps, masks = collect_grpo_group(image_id, ground_truth, group_size, max_tool_calls)
        advantages = compute_group_advantages(rewards)
        groups.append((trajs, advantages, old_lps, masks))
        all_rewards.extend(rewards)

    n_total_rollouts = len(image_ids_and_gts) * group_size

    # At epoch 0, ratio should be close to (not exactly) 1 -- both old and new log-probs
    # come from essentially the same weights. Small deviation from exactly 1 is expected
    # and comes from LoRA dropout being active during this train-mode forward pass but not
    # during collect_grpo_group's eval-mode one; that's noise in this simplified
    # single-model-instance design, not a bug to chase.
    model.train()
    for epoch in range(epochs_per_batch):
        optimizer.zero_grad()
        for trajs, advantages, old_lps, masks in groups:
            for traj, advantage, old_lp, mask in zip(trajs, advantages, old_lps, masks):
                enc = build_full_trajectory_labels(traj)
                enc = {k: v.to(model.device) for k, v in enc.items()}
                new_lp, _ = compute_token_log_probs(enc, use_reference=False)

                ratio = torch.exp(new_lp - old_lp.to(model.device))
                adv = advantage.to(model.device)
                unclipped, clipped = ratio * adv, torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv
                per_token_loss = -torch.min(unclipped, clipped)

                if kl_beta > 0:
                    with torch.no_grad():
                        ref_lp, _ = compute_token_log_probs(enc, use_reference=True)
                    # k3 estimator (Schulman): always >= 0, lower variance than the naive
                    # log-ratio -- the standard choice in the GRPO paper.
                    log_ratio_ref = ref_lp - new_lp
                    per_token_kl = torch.exp(log_ratio_ref) - log_ratio_ref - 1
                    per_token_loss = per_token_loss + kl_beta * per_token_kl

                # Normalize by n_total_rollouts so accumulated gradients are an average,
                # not a sum -- otherwise effective step size scales with batch size.
                loss = (per_token_loss * mask).sum() / mask.sum().clamp(min=1) / n_total_rollouts
                loss.backward()
        optimizer.step()

    model.eval()
    return {
        "mean_reward": sum(all_rewards) / max(len(all_rewards), 1),
        "n_rollouts": len(all_rewards),
        "epochs_per_batch": epochs_per_batch, "clip_eps": clip_eps, "kl_beta": kl_beta,
    }


print(
    "grpo_step() is defined but NOT run automatically -- it's expensive (group_size x "
    "len(image_ids_and_gts) x epochs_per_batch full forward/backward passes per call) and "
    "should be smoke-tested deliberately, e.g.:\n\n"
    "    sample_id = images_df.dropna(subset=['local_path'])['id'].iloc[0]\n"
    "    gt = tool_locate_abnormal_teeth(sample_id)[0]\n"
    "    stats = grpo_step([(sample_id, gt)], group_size=2, epochs_per_batch=1)  # tiny first try\n"
    "    print(stats)\n\n"
    "Start with group_size=2-4, epochs_per_batch=1, on Kaggle/Colab free tier; only raise "
    "group_size or epochs_per_batch once you've confirmed memory headroom (section 7's "
    "estimator, which doesn't yet account for the reference-policy forward pass -- budget "
    "some extra margin) on the 4090."
)


### Training-curve logging

A single before/after number isn't what an RL paper's results section runs on -- reviewers
expect to see whether the policy actually improved over training, and how noisy that
improvement was. `grpo_step()` itself stays a pure function (easier to test); logging is a
thin wrapper you call around it.


In [ ]:
GRPO_LOG_PATH = os.path.join(DATA_DIR, "grpo_training_log.jsonl")


def log_grpo_step(stats, extra=None):
    """Append one grpo_step() call's stats to a persistent log -- call this after every
    grpo_step() call that's an actual training step, not a smoke test, or the log becomes
    noise you have to filter out later."""
    record = {**stats, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), **(extra or {})}
    with open(GRPO_LOG_PATH, "a") as f:
        f.write(json.dumps(to_jsonable(record)) + "\n")
    return record


def plot_grpo_training_curve(log_path=None):
    """Reward-vs-training-step curve -- the figure an RL paper needs, showing whether
    the policy improved (and how noisily) rather than reporting a single before/after
    number with nothing in between."""
    log_path = log_path or GRPO_LOG_PATH
    if not os.path.exists(log_path):
        print(f"No log found at {log_path} yet -- call log_grpo_step() after grpo_step() "
              f"calls to start building one.")
        return None

    records = [json.loads(line) for line in open(log_path) if line.strip()]
    if not records:
        print("Log file exists but is empty.")
        return None

    df = pd.DataFrame(records)
    df["step"] = range(1, len(df) + 1)

    plt.figure(figsize=(8, 4))
    plt.plot(df["step"], df["mean_reward"], marker="o")
    plt.xlabel("grpo_step() call number")
    plt.ylabel("mean reward")
    plt.title("GRPO training progress")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"{len(df)} logged calls. Latest mean_reward: {df['mean_reward'].iloc[-1]:.3f}  "
          f"(first logged: {df['mean_reward'].iloc[0]:.3f})")
    return df


print("Usage pattern for real training runs:\n"
      "  stats = grpo_step([(image_id, gt)], group_size=8)\n"
      "  log_grpo_step(stats, extra={'note': 'first real run'})\n"
      "  # ... repeat for each step ...\n"
      "  plot_grpo_training_curve()")


## 19. Batch trajectory runner (+ the H1 no-tools ablation condition)

Everything so far runs the agent one image at a time. Real evaluation needs this over many
images, with results cached incrementally — a Kaggle/Colab session dying partway through
a held-out-set evaluation run shouldn't mean starting over.


In [ ]:
NO_TOOLS_SYSTEM_PROMPT = (
    "You are a dental radiograph analysis agent. You do NOT have access to any tools -- "
    "reason directly from the single image you are given, in one turn. Respond with "
    'EXACTLY one JSON object: {"final_answer": {"quadrant": <1-4>, "tooth_position": '
    '<1-8>, "diagnosis": "<caries|deep_caries|periapical_lesion|impacted_tooth>", '
    '"confidence": <0-1>}}. Do not include any other text outside the JSON object.'
)


def run_agent_no_tools(image_id, verbose=True):
    """H1 ablation condition (§4.5): the same model and output schema as run_agent(),
    but no tool access and a single reasoning turn over the unmodified image. Returns a
    trajectory shaped identically to run_agent()'s, so combine_reward() and everything
    below works unchanged on either -- run both on the same image_ids to test whether
    tool use actually helps, holding the model fixed."""
    row = images_df[images_df["id"] == image_id].iloc[0]
    image = Image.open(row["local_path"]).convert("RGB")
    messages = [
        {"role": "system", "content": NO_TOOLS_SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": f"Analyze this panoramic X-ray (image_id={image_id})."},
        ]},
    ]
    reply, prompt_len, gen_ids = generate_agent_reply(messages, return_ids=True)
    parsed = parse_agent_json(reply)
    trajectory = {
        "image_id": image_id,
        "turns": [{"raw_output": reply, "parsed": parsed}],
        "tool_calls": 0,
        "final_answer": (parsed or {}).get("final_answer"),
        "format_ok": bool(parsed and "final_answer" in parsed),
        "assistant_token_spans": [{"prompt_len": prompt_len, "token_ids": gen_ids}],
        "messages": messages,
    }
    if verbose:
        print(f"[no-tools] image_id={image_id}: final_answer={trajectory['final_answer']}")
    return trajectory


In [ ]:
def run_agent_batch(image_ids, agent_fn=run_agent, cache_path=None, resume=True):
    """Run `agent_fn` (run_agent or run_agent_no_tools) over image_ids, scoring each
    against its DENTEX ground truth via combine_reward(). Results cache to `cache_path`
    incrementally; resume=True (default) picks up where a prior, interrupted run left off
    instead of re-running everything."""
    results, done_ids = [], set()
    if cache_path and resume and os.path.exists(cache_path):
        with open(cache_path) as f:
            results = json.load(f)
        done_ids = {r["image_id"] for r in results}
        print(f"Resuming: {len(done_ids)} image(s) already evaluated in {cache_path}")

    todo = [i for i in image_ids if i not in done_ids]
    for idx, image_id in enumerate(todo):
        gt_rows = tool_locate_abnormal_teeth(image_id)
        if not gt_rows:
            continue
        ground_truth = {
            "quadrant": gt_rows[0]["quadrant"],
            "tooth_position": gt_rows[0]["tooth_position"],
            "diagnosis": gt_rows[0]["diagnosis"],
        }
        trajectory = agent_fn(image_id, verbose=False)
        total_reward, components = combine_reward(trajectory, ground_truth)
        results.append(to_jsonable({
            "image_id": image_id,
            "ground_truth": ground_truth,
            "final_answer": trajectory["final_answer"],
            "tool_calls": trajectory["tool_calls"],
            "format_ok": trajectory.get("format_ok", False),
            "reward": total_reward,
            "reward_components": components,
        }))
        if cache_path:
            with open(cache_path, "w") as f:
                json.dump(results, f, indent=2)
        if (idx + 1) % 5 == 0 or idx == len(todo) - 1:
            running_mean = sum(r["reward"] for r in results) / len(results)
            print(f"  {idx + 1}/{len(todo)} done (running mean reward: {running_mean:.3f})")

    return results


## 20. Evaluation metrics (§6)

In [ ]:
from sklearn.metrics import f1_score, balanced_accuracy_score


def expected_calibration_error(confidences, correctness, n_bins=10):
    """Standard ECE: bin predictions by stated confidence, compare each bin's mean
    confidence to its actual accuracy, weight by bin size."""
    confidences, correctness = np.array(confidences), np.array(correctness)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(confidences)) * abs(confidences[mask].mean() - correctness[mask].mean())
    return ece


def compute_evaluation_metrics(results):
    """Core §6 metrics from a run_agent_batch() results list: FDI (quadrant + tooth
    position) accuracy, per-diagnosis F1, balanced accuracy, format-compliance rate,
    mean reward, and confidence calibration (ECE) when confidence scores were reported."""
    if not results:
        return {}

    y_true_diag, y_pred_diag = [], []
    fdi_correct = format_ok_count = 0
    confidences, correctness = [], []

    for r in results:
        gt, ans = r["ground_truth"], (r["final_answer"] or {})
        y_true_diag.append(str(gt.get("diagnosis", "")).lower())
        y_pred_diag.append(str(ans.get("diagnosis", "")).lower() if ans else "none")

        quad_ok = ans.get("quadrant") == gt.get("quadrant")
        tooth_ok = ans.get("tooth_position") == gt.get("tooth_position")
        fdi_correct += int(quad_ok and tooth_ok)
        format_ok_count += int(r.get("format_ok", False))

        if ans and "confidence" in ans:
            diag_ok = str(ans.get("diagnosis", "")).lower() == str(gt.get("diagnosis", "")).lower()
            confidences.append(float(ans["confidence"]))
            correctness.append(int(quad_ok and tooth_ok and diag_ok))

    n = len(results)
    labels = sorted(set(y_true_diag) | set(y_pred_diag))
    per_class_f1 = dict(zip(labels, f1_score(y_true_diag, y_pred_diag, labels=labels,
                                              average=None, zero_division=0)))

    metrics = {
        "n_examples": n,
        "fdi_accuracy": fdi_correct / n,
        "diagnosis_balanced_accuracy": balanced_accuracy_score(y_true_diag, y_pred_diag),
        "diagnosis_per_class_f1": per_class_f1,
        "format_compliance_rate": format_ok_count / n,
        "mean_reward": sum(r["reward"] for r in results) / n,
        "expected_calibration_error": (
            expected_calibration_error(confidences, correctness) if len(confidences) >= 5 else None
        ),
    }
    if len(confidences) < 5:
        metrics["_note"] = "fewer than 5 confidence scores available -- ECE skipped"
    return metrics


# Smoke test on whatever's in the demo run above, so this is checked before spending
# real compute on a full batch.
demo_gt = tool_locate_abnormal_teeth(example_id)
if demo_gt:
    demo_reward, demo_components = combine_reward(demo_trajectory, {
        "quadrant": demo_gt[0]["quadrant"], "tooth_position": demo_gt[0]["tooth_position"],
        "diagnosis": demo_gt[0]["diagnosis"],
    })
    demo_metrics = compute_evaluation_metrics([to_jsonable({
        "image_id": example_id,
        "ground_truth": {"quadrant": demo_gt[0]["quadrant"], "tooth_position": demo_gt[0]["tooth_position"],
                          "diagnosis": demo_gt[0]["diagnosis"]},
        "final_answer": demo_trajectory["final_answer"], "tool_calls": demo_trajectory["tool_calls"],
        "format_ok": demo_trajectory.get("format_ok", False),
        "reward": demo_reward, "reward_components": demo_components,
    })])
    print(json.dumps(demo_metrics, indent=2, default=str))


### A floor to compare against: majority-class baseline

Any trained agent needs to clearly beat this to mean anything. Computed from the TRAINING
pool only (never `holdout_ids`), then scored against real held-out ground truth.


In [ ]:
def majority_class_baseline_metrics(holdout_image_ids):
    """Always predicts the single most common quadrant, tooth position, and diagnosis
    (measured on the training pool) -- a naive floor for the §6 metrics above."""
    train_annots = annots_df[~annots_df["image_id"].isin(holdout_ids)]
    majority_quadrant = train_annots["category_id_1"].mode().iloc[0]
    majority_tooth = train_annots["category_id_2"].mode().iloc[0]
    diag_lookup = dict(zip(categories_df["id"], categories_df["name"])) if len(categories_df) else {}
    majority_diag_id = train_annots[diag_col].mode().iloc[0] if diag_col else None
    majority_diag = diag_lookup.get(majority_diag_id, "unknown")

    fake_results = []
    for image_id in holdout_image_ids:
        gt_rows = tool_locate_abnormal_teeth(image_id)
        if not gt_rows:
            continue
        gt = {"quadrant": gt_rows[0]["quadrant"], "tooth_position": gt_rows[0]["tooth_position"],
              "diagnosis": gt_rows[0]["diagnosis"]}
        fake_results.append(to_jsonable({
            "image_id": image_id, "ground_truth": gt,
            "final_answer": {"quadrant": majority_quadrant, "tooth_position": majority_tooth,
                              "diagnosis": majority_diag, "confidence": 1.0},
            "tool_calls": 0, "format_ok": True, "reward": 0.0, "reward_components": {},
        }))

    metrics = compute_evaluation_metrics(fake_results)
    print(f"Majority-class baseline (always predicts quadrant={majority_quadrant}, "
          f"tooth_position={majority_tooth}, diagnosis={majority_diag!r}):")
    print(f"  fdi_accuracy={metrics.get('fdi_accuracy'):.3f}  "
          f"balanced_accuracy={metrics.get('diagnosis_balanced_accuracy'):.3f}")
    return metrics


print("Ready -- run once you have some held-out ground truth to compare against, e.g.:\n"
      "  majority_class_baseline_metrics(list(holdout_ids)[:20])")


### Zero-shot external VLM baseline (§6, baseline #1)

The one named baseline that had no implementation until now: a general-purpose VLM (GPT-4o
or similar), prompted zero-shot -- no fine-tuning, no tools, single pass. This is the exact
condition the whole proposal is positioned against (§3.3's cited GPT-4o dental studies), so
it needs its own number, not just an assumption that it performs the way the literature says
it does on a *different* dataset. Reuses `call_llm` from section 16 rather than a new client.


In [ ]:
ZERO_SHOT_PROMPT = (
    'You are looking at a panoramic dental X-ray. Identify ONE abnormal tooth and respond '
    'with exactly one JSON object: {"quadrant": <1-4>, "tooth_position": <1-8>, "diagnosis": '
    '"<caries|deep_caries|periapical_lesion|impacted_tooth>", "confidence": <0-1>}. '
    "No other text, no tools, no reasoning shown -- exactly what a single zero-shot API "
    "call would receive in the studies section 3.3 describes."
)


def run_zero_shot_baseline(image_ids, provider="openai", model="gpt-4o", cache_path=None, resume=True):
    """No fine-tuning, no tool access, single pass -- deliberately as close as this
    notebook can get to the exact zero-shot condition the cited dental GPT-4V/4o studies
    report. Any provider/model works as long as call_llm (section 16) supports it."""
    results, done_ids = [], set()
    if cache_path and resume and os.path.exists(cache_path):
        with open(cache_path) as f:
            results = json.load(f)
        done_ids = {r["image_id"] for r in results}
        print(f"Resuming: {len(done_ids)} image(s) already processed in {cache_path}")

    for image_id in image_ids:
        if image_id in done_ids:
            continue
        gt_rows = tool_locate_abnormal_teeth(image_id)
        if not gt_rows:
            continue
        ground_truth = {"quadrant": gt_rows[0]["quadrant"], "tooth_position": gt_rows[0]["tooth_position"],
                         "diagnosis": gt_rows[0]["diagnosis"]}
        row = images_df[images_df["id"] == image_id].iloc[0]
        image = Image.open(row["local_path"]).convert("RGB")

        raw = call_llm(provider, model, ZERO_SHOT_PROMPT, "Analyze this X-ray.", image=image)
        parsed = parse_agent_json(raw)

        results.append(to_jsonable({
            "image_id": image_id, "ground_truth": ground_truth, "final_answer": parsed,
            "tool_calls": 0, "format_ok": parsed is not None,
            "reward": reward_accuracy({"final_answer": parsed}, ground_truth) if parsed else 0.0,
            "reward_components": {},
        }))
        if cache_path:
            with open(cache_path, "w") as f:
                json.dump(results, f, indent=2)

    return results


print("run_zero_shot_baseline(image_ids) ready. Example, once OPENAI_API_KEY is set:\n"
      "  zs_results = run_zero_shot_baseline(list(holdout_ids)[:10],\n"
      "                                       cache_path=os.path.join(DATA_DIR, 'zero_shot_eval.json'))\n"
      "  print(compute_evaluation_metrics(zs_results))")


## 21. Ablations: H1 (tools vs. no tools) and checkpoint comparison (H2)

Directly operationalizes the proposal's two primary hypotheses (§4.5): does tool access help
(H1), and does RL training help beyond SFT alone (H2)? Both run the small evaluation harness
above, just varying which agent function or which checkpoint is active.


In [ ]:
def bootstrap_paired_diff_ci(values_a, values_b, n_boot=2000, ci=0.95, seed=0):
    """95% bootstrap CI for the mean paired difference (a - b), e.g. reward with tools
    minus reward without tools, matched by image. If the interval excludes zero, the
    difference is unlikely to be due to chance alone at this sample size -- if it includes
    zero, the sample is too small (or the effect too weak) to conclude much yet."""
    rng = np.random.default_rng(seed)
    diffs = np.array(values_a) - np.array(values_b)
    boot_means = [rng.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(n_boot)]
    lo, hi = np.percentile(boot_means, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return diffs.mean(), (lo, hi)


def run_h1_ablation(image_ids):
    """H1: same model, with vs. without tool access, on the same images."""
    with_tools = run_agent_batch(image_ids, agent_fn=run_agent,
                                  cache_path=os.path.join(DATA_DIR, "ablation_with_tools.json"))
    without_tools = run_agent_batch(image_ids, agent_fn=run_agent_no_tools,
                                     cache_path=os.path.join(DATA_DIR, "ablation_without_tools.json"))
    m_with, m_without = compute_evaluation_metrics(with_tools), compute_evaluation_metrics(without_tools)

    print("H1 ablation -- same model, with vs. without tool access:")
    for key in ("fdi_accuracy", "diagnosis_balanced_accuracy", "format_compliance_rate", "mean_reward"):
        print(f"  {key:28s}  with_tools={m_with.get(key):.3f}   without_tools={m_without.get(key):.3f}")

    # Paired bootstrap CI on the reward difference, matched by image_id -- a bigger mean
    # alone doesn't say whether the gap is likely real at this sample size; this does.
    by_id_with = {r["image_id"]: r["reward"] for r in with_tools}
    by_id_without = {r["image_id"]: r["reward"] for r in without_tools}
    common_ids = sorted(set(by_id_with) & set(by_id_without))
    if len(common_ids) >= 5:
        mean_diff, (ci_lo, ci_hi) = bootstrap_paired_diff_ci(
            [by_id_with[i] for i in common_ids], [by_id_without[i] for i in common_ids]
        )
        verdict = ("CI excludes 0: difference unlikely due to chance at this sample size."
                   if (ci_lo > 0 or ci_hi < 0) else
                   "CI includes 0: not yet distinguishable from no difference -- evaluate "
                   "more images before concluding much either way.")
        print(f"\nPaired reward difference (with - without), n={len(common_ids)}: "
              f"{mean_diff:.3f}  [95% CI: {ci_lo:.3f}, {ci_hi:.3f}]\n  -> {verdict}")
    else:
        print(f"\nOnly {len(common_ids)} images have both conditions -- too few for a "
              f"meaningful CI (want at least ~30 for the real evaluation run).")

    return m_with, m_without


In [ ]:
def compare_checkpoints(checkpoint_tags, image_ids):
    """H2 (and general model comparison): reload each named checkpoint (e.g. a fresh
    base marker, 'sft-3b-v1', a later GRPO tag from save_checkpoint) and evaluate all of
    them on the same image_ids. Restores whatever model/processor were loaded before this
    ran, once finished. On 16GB free-tier GPUs, reloading several checkpoints back to back
    can get tight on memory -- torch.cuda.empty_cache() is called between each."""
    global model, processor
    original_model, original_processor = model, processor
    comparison = {}
    try:
        for tag in checkpoint_tags:
            if tag != "current (no reload)":
                ckpt_dir = os.path.join(CHECKPOINT_DIR, tag)
                model = ModelClass.from_pretrained(ckpt_dir, quantization_config=bnb_config, device_map="auto")
                processor = AutoProcessor.from_pretrained(ckpt_dir)
            results = run_agent_batch(
                image_ids, agent_fn=run_agent,
                cache_path=os.path.join(DATA_DIR, f"eval_{tag.replace(' ', '_')}.json"),
            )
            comparison[tag] = compute_evaluation_metrics(results)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    finally:
        model, processor = original_model, original_processor

    print("Checkpoint comparison:")
    for tag, m in comparison.items():
        print(f"  {tag}: fdi_accuracy={m.get('fdi_accuracy'):.3f}  "
              f"balanced_accuracy={m.get('diagnosis_balanced_accuracy'):.3f}  "
              f"mean_reward={m.get('mean_reward'):.3f}")
    return comparison


print("Both ablation functions are defined and ready. Example usage once you have a real "
      "held-out sample and at least one saved checkpoint (start tiny -- 5-10 images -- "
      "before scaling to the full held-out set):\n")
print("  sample_ids = list(holdout_ids)[:10]  # draw from the held-out set, never the training pool")
print("  run_h1_ablation(sample_ids)")
print("  compare_checkpoints(['current (no reload)', 'sft-3b-v1'], sample_ids)")


### Reward-weight sensitivity sweep (§5.5)

The proposal is explicit that the reward weights (`acc`/`fmt`/`tool`/`eff`) are hyperparameters
to sweep and report, not fixed by assumption. This re-scores the SAME collected trajectories
under each weight combination via `combine_reward` -- one round of agent rollouts, many
weightings -- rather than re-running the (expensive) agent once per combination.


In [ ]:
def sweep_reward_weights(image_ids, weight_grid, agent_fn=run_agent):
    """Collects trajectories once, then rescoring them under each weights dict in
    weight_grid via combine_reward -- only the weighting changes per row, not the
    underlying reward components, so this is cheap relative to a fresh agent run per
    combination."""
    trajectories_and_gts = []
    for image_id in image_ids:
        gt_rows = tool_locate_abnormal_teeth(image_id)
        if not gt_rows:
            continue
        ground_truth = {"quadrant": gt_rows[0]["quadrant"], "tooth_position": gt_rows[0]["tooth_position"],
                         "diagnosis": gt_rows[0]["diagnosis"]}
        trajectories_and_gts.append((agent_fn(image_id, verbose=False), ground_truth))

    rows = []
    for weights in weight_grid:
        totals = [combine_reward(traj, gt, weights=weights)[0] for traj, gt in trajectories_and_gts]
        rows.append({**weights, "mean_reward": sum(totals) / max(len(totals), 1), "n": len(totals)})
    return pd.DataFrame(rows)


DEFAULT_WEIGHT_GRID = [
    {"acc": 1.0, "fmt": 0.2, "tool": 0.2, "eff": 0.1},   # the proposal's default (§5.5)
    {"acc": 1.0, "fmt": 0.0, "tool": 0.0, "eff": 0.0},   # accuracy only, as a reference point
    {"acc": 1.0, "fmt": 0.1, "tool": 0.4, "eff": 0.1},   # weight tool-use more heavily
    {"acc": 1.0, "fmt": 0.4, "tool": 0.1, "eff": 0.1},   # weight format compliance more heavily
    {"acc": 0.7, "fmt": 0.2, "tool": 0.2, "eff": 0.1},   # de-emphasize accuracy slightly
]

print("sweep_reward_weights(image_ids, DEFAULT_WEIGHT_GRID) ready. Example:\n"
      "  sample_ids = list(holdout_ids)[:10]\n"
      "  print(sweep_reward_weights(sample_ids, DEFAULT_WEIGHT_GRID))")


## 22. Reasoning-grounding check (R_judge) at evaluation time

§5.5/§6's R_judge, applied here as an evaluation metric to *real agent trajectories* (not the
synthetic training traces section 16 verifies) -- does the model's own stated reasoning stay
grounded in the tool outputs it actually cited, or does it hallucinate evidence it never saw?
Reuses the two-model-family setup from section 16 so the judge isn't grading its own mistakes.


In [ ]:
TRAJECTORY_JUDGE_SYSTEM_PROMPT = (
    "You are a strict verifier, not a rewriter. You will see an X-ray image, the KNOWN "
    "correct answer, and an agent's full reasoning + tool-call trace (not synthetic -- "
    "this is a real trajectory from a live agent). Judge ONLY whether every claim the "
    "agent makes is actually supported by the image, its own tool outputs, and the known "
    'answer -- reject anything asserted without support, even if the final answer happens '
    'to be correct. Respond with exactly one JSON object: {"grounded": true/false, "reason": "..."}.'
)


def reward_judge(image, ground_truth, trajectory):
    """Evaluation-time R_judge for one trajectory. Costs one API call -- keep sample
    sizes small for a first pass (see evaluate_reasoning_grounding below)."""
    reasoning_text = "\n".join(t["raw_output"] for t in trajectory["turns"])
    user_content = (
        f"Known correct answer: {json.dumps(ground_truth)}\n\n"
        f"Agent's full reasoning + tool-call trace:\n{reasoning_text}\n\n"
        f"Agent's final answer: {json.dumps(trajectory.get('final_answer'))}"
    )
    raw = call_llm(VERIFIER_PROVIDER, VERIFIER_MODEL, TRAJECTORY_JUDGE_SYSTEM_PROMPT, user_content, image=image)
    parsed = parse_agent_json(raw)
    return parsed if parsed else {"grounded": False, "reason": "judge output unparseable"}


def evaluate_reasoning_grounding(image_ids, sample_n=10):
    """Run R_judge over a capped sample of trajectories and report the grounded rate."""
    if not (os.environ.get("OPENAI_API_KEY") and os.environ.get("ANTHROPIC_API_KEY")):
        print("Set API keys (see section 16) to run this -- skipping.")
        return None

    grounded_flags = []
    for image_id in image_ids[:sample_n]:
        gt_rows = tool_locate_abnormal_teeth(image_id)
        if not gt_rows:
            continue
        ground_truth = {"quadrant": gt_rows[0]["quadrant"], "tooth_position": gt_rows[0]["tooth_position"],
                         "diagnosis": gt_rows[0]["diagnosis"]}
        row = images_df[images_df["id"] == image_id].iloc[0]
        image = Image.open(row["local_path"]).convert("RGB")
        trajectory = run_agent(image_id, verbose=False)
        verdict = reward_judge(image, ground_truth, trajectory)
        grounded_flags.append(bool(verdict.get("grounded")))
        print(f"  image_id={image_id}: grounded={verdict.get('grounded')}  ({verdict.get('reason', '')})")

    rate = sum(grounded_flags) / len(grounded_flags) if grounded_flags else None
    print(f"\nReasoning-grounding rate over {len(grounded_flags)} sampled trajectories: {rate}")
    return rate


## 23. Trajectory visualizer (qualitative inspection)

Walks a trajectory's actual message history (not a summary) so you see exactly what the
model saw at each turn, including tool-result crops -- useful for debugging and as a figure
source later.


In [ ]:
def visualize_trajectory(trajectory):
    panels = []
    for msg in trajectory["messages"]:
        if msg["role"] == "system":
            continue
        content = msg["content"] if isinstance(msg["content"], list) else [{"type": "text", "text": msg["content"]}]
        img = next((c["image"] for c in content if c.get("type") == "image"), None)
        text = " ".join(c["text"] for c in content if c.get("type") == "text")
        panels.append((f"[{msg['role']}] {text[:100]}", img))

    imgs_only = [(label, img) for label, img in panels if img is not None]
    if not imgs_only:
        print("No images found in this trajectory's message history.")
        return

    fig, axes = plt.subplots(1, len(imgs_only), figsize=(5 * len(imgs_only), 5))
    if len(imgs_only) == 1:
        axes = [axes]
    for ax, (label, img) in zip(axes, imgs_only):
        ax.imshow(img)
        ax.set_title(label, fontsize=8, wrap=True)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("\nFull turn-by-turn text:")
    for label, _ in panels:
        print(" -", label)


visualize_trajectory(demo_trajectory)


## 24. Stage 0: fine-tune a real grounding detector (replaces the oracle)

Everything from section 12 onward has been using `tool_locate_abnormal_teeth`'s oracle —
DENTEX's own ground truth, wearing the interface a real tool would have. This section trains
an actual detector on the quadrant-enumeration data (634 images) to replace it. Deliberately
kept minimal (a light backbone, few epochs, no LR schedule) so it *runs* on
Kaggle/Colab free-tier GPUs — treat its first output the way you'd treat any freshly-trained
detector: check it qualitatively before trusting it, not as a finished Stage 0.

One design choice worth noting: this detector predicts quadrant + tooth position only, **not**
diagnosis — diagnosis isn't in the quadrant-enumeration file it trains on, and more importantly
that's the right division of labor per the proposal (§5.3): the tool locates and names a tooth,
the VLM's own reasoning (after zooming in) is what determines the diagnosis. The oracle
currently "cheats" by also returning ground-truth diagnosis for development convenience —
`tool_locate_abnormal_teeth_learned` below does not, on purpose.


In [ ]:
import torchvision
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader


def flip_quadrant(quadrant):
    """A horizontal flip swaps anatomical left/right, so under FDI notation quadrant
    1<->2 (upper right/left) and 3<->4 (lower right/left) swap too -- it is NOT enough to
    just mirror the pixels and boxes; the quadrant label itself has to swap with them, or
    every flipped training example silently teaches the detector the wrong quadrant. Tooth
    position within a quadrant (1=central incisor .. 8=third molar, counting outward from
    the midline) does NOT change under a flip -- only which side of the midline that
    position sits on does."""
    return {1: 2, 2: 1, 3: 4, 4: 3}[quadrant]


class DentexDetectionDataset(Dataset):
    """Wraps DENTEX quadrant-enumeration annotations as a torchvision detection dataset:
    each abnormal tooth becomes one box, labeled by its FDI position (quadrant, tooth_position
    combined into a single class id 1-32; class 0 is background, reserved by torchvision's
    convention). With augment=True, applies a horizontal flip to ~50% of examples (with the
    quadrant relabeling above) -- cheap, and doubles the effective training signal from a
    training pool small enough (a few hundred images) that this genuinely helps."""

    def __init__(self, images_df_in, annots_df_in, augment=False):
        self.image_ids = sorted(annots_df_in["image_id"].unique())
        self.images_lookup = images_df_in.set_index("id")
        self.annots_df = annots_df_in
        self.augment = augment

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        row = self.images_lookup.loc[image_id]
        image = Image.open(row["local_path"]).convert("RGB")
        anns = self.annots_df[self.annots_df["image_id"] == image_id]

        boxes, labels = [], []
        for _, ann in anns.iterrows():
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            boxes.append([x, y, x + w, y + h])
            quad = int(ann.get("category_id_1", 1))
            tooth = int(ann.get("category_id_2", 1))
            labels.append((quad - 1) * 8 + tooth)  # 1-32; 0 reserved for background

        image_tensor = torchvision.transforms.functional.to_tensor(image)

        if self.augment and random.random() < 0.5 and boxes:
            image_tensor = torchvision.transforms.functional.hflip(image_tensor)
            img_w = image_tensor.shape[-1]
            flipped_boxes, flipped_labels = [], []
            for (x1, y1, x2, y2), label in zip(boxes, labels):
                flipped_boxes.append([img_w - x2, y1, img_w - x1, y2])
                quad, tooth = (label - 1) // 8 + 1, (label - 1) % 8 + 1
                new_quad = flip_quadrant(quad)
                flipped_labels.append((new_quad - 1) * 8 + tooth)
            boxes, labels = flipped_boxes, flipped_labels

        boxes_t = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        labels_t = torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64)
        target = {"boxes": boxes_t, "labels": labels_t, "image_id": torch.tensor([image_id])}
        return image_tensor, target


def detection_collate_fn(batch):
    return tuple(zip(*batch))


def build_stage0_detector(num_classes=33):  # 32 FDI positions + background
    detector = fasterrcnn_mobilenet_v3_large_fpn(weights="DEFAULT")
    in_features = detector.roi_heads.box_predictor.cls_score.in_features
    detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return detector


In [ ]:
def train_stage0_detector(epochs=1, batch_size=2, lr=5e-4, subset_n=None, verbose_every=10, augment=True):
    """Minimal Stage-0 training loop. Trains on the *non-held-out* images only -- same
    `holdout_ids` boundary as everything else in this notebook, so the detector is never
    trained on what section 20+ later evaluates against."""
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_ids = set(images_df[~images_df["id"].isin(holdout_ids)]["id"])
    train_annots = annots_df[annots_df["image_id"].isin(train_ids)]
    dataset = DentexDetectionDataset(images_df[images_df["local_path"].notna()], train_annots, augment=augment)
    if subset_n:
        dataset.image_ids = dataset.image_ids[:subset_n]
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=detection_collate_fn)

    detector = build_stage0_detector().to(device)
    optimizer = torch.optim.AdamW([p for p in detector.parameters() if p.requires_grad], lr=lr)

    detector.train()
    for epoch in range(epochs):
        running_loss, n_batches = 0.0, 0
        for step, (images, targets) in enumerate(loader):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = detector(images, targets)
            loss = sum(loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            n_batches += 1
            if (step + 1) % verbose_every == 0:
                print(f"  epoch {epoch + 1} step {step + 1}/{len(loader)}  loss={loss.item():.3f}")
        print(f"Epoch {epoch + 1}/{epochs} done -- mean loss: {running_loss / max(n_batches, 1):.3f}")

    return detector


def tool_locate_abnormal_teeth_learned(image_id, detector, score_threshold=0.5):
    """Same return shape as the oracle tool_locate_abnormal_teeth(), but backed by a real
    trained detector. `diagnosis` is always None here (see the note above section 24) --
    that's intentional, not a missing feature."""
    row = images_df[images_df["id"] == image_id].iloc[0]
    image = Image.open(row["local_path"]).convert("RGB")
    device = next(detector.parameters()).device
    image_tensor = torchvision.transforms.functional.to_tensor(image).to(device)

    detector.eval()
    with torch.no_grad():
        prediction = detector([image_tensor])[0]

    results = []
    for box, label, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"]):
        if score < score_threshold:
            continue
        label = int(label.item())
        quadrant, tooth_position = (label - 1) // 8 + 1, (label - 1) % 8 + 1
        x1, y1, x2, y2 = box.tolist()
        results.append({
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "quadrant": quadrant, "tooth_position": tooth_position,
            "fdi_label": tool_fdi_label(quadrant, tooth_position),
            "score": float(score.item()),
            "diagnosis": None,
        })
    return results


print("build_stage0_detector() / train_stage0_detector() are ready. Sanity-check on a tiny "
      "subset before a real run, e.g.:\n"
      "  detector = train_stage0_detector(epochs=1, subset_n=20)")


In [ ]:
def visualize_detector_predictions(detector, image_id, score_threshold=0.5):
    """Red = the detector's predictions, green = ground truth, on the same image --
    the fastest way to tell whether a freshly-trained Stage-0 detector is worth trusting
    before wiring it into the agent loop in place of the oracle."""
    predictions = tool_locate_abnormal_teeth_learned(image_id, detector, score_threshold)
    ground_truth = tool_locate_abnormal_teeth(image_id)

    row = images_df[images_df["id"] == image_id].iloc[0]
    img = Image.open(row["local_path"]).convert("RGB")
    draw = ImageDraw.Draw(img)
    for p in predictions:
        x, y, w, h = p["bbox"]
        draw.rectangle([x, y, x + w, y + h], outline="red", width=3)
        draw.text((x, max(0, y - 12)), f"pred {p['fdi_label']} ({p['score']:.2f})", fill="red")
    for g in ground_truth:
        x, y, w, h = g["bbox"]
        draw.rectangle([x, y, x + w, y + h], outline="lime", width=2)
        draw.text((x, y + h + 2), f"gt {g['fdi_label']}", fill="lime")

    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.title(f"image_id={image_id}: red=predicted, green=ground truth")
    plt.axis("off")
    plt.show()


print("visualize_detector_predictions(detector, image_id) ready once a detector is trained.")


### Quantitative Stage-0 evaluation

`visualize_detector_predictions` is qualitative -- useful, but "eyeball a few images" doesn't
scale and doesn't give you a number to put in a table. This adds precision/recall/F1 via
greedy IoU matching, mirroring the train→evaluate pattern already used for the VLM itself
(sections 19-23) rather than leaving Stage 0 as the one piece of the pipeline nothing measures.


In [ ]:
def compute_iou(box_a, box_b):
    """IoU between two [x1, y1, x2, y2] boxes."""
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b
    ix1, iy1 = max(xa1, xb1), max(ya1, yb1)
    ix2, iy2 = min(xa2, xb2), min(ya2, yb2)
    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, xa2 - xa1) * max(0.0, ya2 - ya1)
    area_b = max(0.0, xb2 - xb1) * max(0.0, yb2 - yb1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


def evaluate_stage0_detector(detector, image_ids, iou_threshold=0.5, score_threshold=0.5):
    """Simplified single-operating-point precision/recall/F1 via greedy IoU matching --
    honest and correct at this one (IoU, score) threshold pair, not a substitute for a full
    COCO-style mAP evaluator across multiple thresholds if you need one for a paper table. A
    predicted box counts correct only if it clears the IoU threshold against an unmatched
    ground-truth box AND agrees on quadrant + tooth position."""
    tp = fp = fn = 0
    for image_id in image_ids:
        predictions = sorted(
            tool_locate_abnormal_teeth_learned(image_id, detector, score_threshold),
            key=lambda p: -p["score"],
        )
        ground_truth = tool_locate_abnormal_teeth(image_id)
        gt_boxes = [
            (g["quadrant"], g["tooth_position"],
             [g["bbox"][0], g["bbox"][1], g["bbox"][0] + g["bbox"][2], g["bbox"][1] + g["bbox"][3]])
            for g in ground_truth
        ]
        matched = set()

        for p in predictions:
            p_box = [p["bbox"][0], p["bbox"][1], p["bbox"][0] + p["bbox"][2], p["bbox"][1] + p["bbox"][3]]
            best_iou, best_j = 0.0, -1
            for j, (_, _, gbox) in enumerate(gt_boxes):
                if j in matched:
                    continue
                iou = compute_iou(p_box, gbox)
                if iou > best_iou:
                    best_iou, best_j = iou, j

            is_correct = (
                best_j >= 0 and best_iou >= iou_threshold
                and gt_boxes[best_j][0] == p["quadrant"] and gt_boxes[best_j][1] == p["tooth_position"]
            )
            if is_correct:
                tp += 1
                matched.add(best_j)
            else:
                fp += 1
        fn += len(gt_boxes) - len(matched)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    print(f"Stage 0 detector @ IoU>={iou_threshold}, score>={score_threshold}: "
          f"precision={precision:.3f}  recall={recall:.3f}  f1={f1:.3f}  (tp={tp} fp={fp} fn={fn})")
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}


print("evaluate_stage0_detector(detector, image_ids) ready once a detector is trained -- "
      "evaluate on holdout_ids, never the training pool, e.g.:\n"
      "  evaluate_stage0_detector(detector, list(holdout_ids)[:20])")


## 25. Results reporting (paper-shaped tables)

Turns raw metrics dicts into one comparison table and saves both a machine-readable JSON and
a Markdown version close to drop-in for a results section.


In [ ]:
def metrics_to_dataframe(named_metrics):
    rows = []
    for name, m in named_metrics.items():
        rows.append({
            "condition": name, "n": m.get("n_examples"),
            "fdi_accuracy": m.get("fdi_accuracy"),
            "balanced_accuracy": m.get("diagnosis_balanced_accuracy"),
            "format_compliance": m.get("format_compliance_rate"),
            "mean_reward": m.get("mean_reward"),
            "ECE": m.get("expected_calibration_error"),
        })
    return pd.DataFrame(rows).set_index("condition")


def save_results_report(named_metrics, path=None):
    path = path or os.path.join(DATA_DIR, "results_report")
    df = metrics_to_dataframe(named_metrics)
    df.to_json(path + ".json", orient="index", indent=2)
    with open(path + ".md", "w") as f:
        f.write("# Evaluation results\n\n")
        f.write(df.to_markdown())
    print(f"Saved {path}.json and {path}.md")
    return df


print("metrics_to_dataframe() / save_results_report() ready -- feed them a "
      "{condition_name: metrics_dict} mapping, e.g. the output of section 26 below.")


### Failure-mode breakdown

Aggregate accuracy alone won't satisfy a reviewer -- they'll want to know *how* the agent
fails, not just how often. This categorizes each result from `run_agent_batch()` (or
`run_zero_shot_baseline()`, same shape) into one of a few buckets, the basis for a
failure-analysis table in the paper.


In [ ]:
def categorize_failure(result):
    """Tags one evaluation result into a failure category."""
    if not result.get("format_ok"):
        return "format_failure"
    ans = result.get("final_answer") or {}
    gt = result["ground_truth"]
    quad_ok = ans.get("quadrant") == gt.get("quadrant")
    tooth_ok = ans.get("tooth_position") == gt.get("tooth_position")
    diag_ok = str(ans.get("diagnosis", "")).lower() == str(gt.get("diagnosis", "")).lower()

    if quad_ok and tooth_ok and diag_ok:
        return "correct"
    if quad_ok and tooth_ok and not diag_ok:
        return "wrong_diagnosis_right_location"
    if not (quad_ok and tooth_ok) and diag_ok:
        return "right_diagnosis_wrong_location"
    return "wrong_location_and_diagnosis"


def failure_mode_breakdown(results):
    """Counts + fraction per category across a results list -- directly the table for a
    failure-analysis section, not just an aggregate number."""
    categories = [categorize_failure(r) for r in results]
    counts = pd.Series(categories).value_counts()
    breakdown = pd.DataFrame({"count": counts, "fraction": counts / len(categories)})
    print(breakdown)
    return breakdown


print("categorize_failure() / failure_mode_breakdown() ready -- feed either a "
      "run_agent_batch() results list or a run_zero_shot_baseline() one (same shape), e.g.:\n"
      "  failure_mode_breakdown(with_tools)  # from run_h1_ablation's return values")


## 26. Run the full evaluation suite (one call)

Ties sections 20-22 and 25 together: majority baseline, H1 ablation, an optional checkpoint
comparison, and an optional R_judge grounding check, then saves a combined report. The
function to reach for once a checkpoint actually exists and is worth evaluating seriously.


In [ ]:
def run_full_evaluation_suite(image_ids, checkpoint_tags=("current (no reload)",),
                               run_judge=True, judge_sample_n=5):
    report = {"majority_baseline": majority_class_baseline_metrics(image_ids)}

    m_with, m_without = run_h1_ablation(image_ids)
    report["with_tools"], report["without_tools"] = m_with, m_without

    if len(checkpoint_tags) > 1 or checkpoint_tags[0] != "current (no reload)":
        report.update(compare_checkpoints(list(checkpoint_tags), image_ids))

    if run_judge:
        report["_reasoning_grounding_rate"] = evaluate_reasoning_grounding(image_ids, sample_n=judge_sample_n)

    scoreable = {k: v for k, v in report.items() if isinstance(v, dict) and "n_examples" in v}
    df = save_results_report(scoreable)
    print(df)
    return report, df


print("run_full_evaluation_suite(image_ids) ready. Example, once you have real ground truth "
      "and at least one saved checkpoint:\n"
      "  sample_ids = list(holdout_ids)[:10]\n"
      "  report, df = run_full_evaluation_suite(sample_ids)")


## 27. Cross-dataset generalization (§6) — fill in once you have a path

The secondary 1,512-image panoramic dataset's exact public download location wasn't confirmed
while building this notebook — its source paper describes it as publicly available, but a
direct link wasn't verified, so guessing one risked pointing you at something wrong. Check the
paper's Data Availability statement, or search Kaggle/Hugging Face for its condition list
(fillings, cavities, implants, impacted teeth — 11,137 annotations) to find the real location.

Everything below is otherwise ready to run once `SECOND_DATASET_PATH` is filled in and the
block uncommented — `run_agent_batch()` / `compute_evaluation_metrics()` are already
dataset-agnostic; the only real work is getting a second dataset into the same
`images_df`/`annots_df`/`local_path` shape section 1 builds for DENTEX.


In [ ]:
SECOND_DATASET_PATH = None  # <- fill in: a local folder path, or a Hugging Face repo_id

# if SECOND_DATASET_PATH:
#     second_coco = load_coco_json(os.path.join(SECOND_DATASET_PATH, "annotations.json"))
#     second_images_df = pd.DataFrame(second_coco["images"])
#     second_annots_df = pd.DataFrame(second_coco["annotations"])
#     second_annots_df["bbox"] = second_annots_df["bbox"].apply(list)
#
#     # Avoid id collisions with DENTEX's own ids before merging.
#     id_offset = int(images_df["id"].max()) + 100000
#     second_images_df["id"] = second_images_df["id"] + id_offset
#     second_annots_df["image_id"] = second_annots_df["image_id"] + id_offset
#
#     second_image_files = (
#         glob.glob(os.path.join(SECOND_DATASET_PATH, "**", "*.png"), recursive=True)
#         + glob.glob(os.path.join(SECOND_DATASET_PATH, "**", "*.jpg"), recursive=True)
#     )
#     second_by_basename = {os.path.basename(p): p for p in second_image_files}
#     second_images_df["local_path"] = second_images_df["file_name"].apply(
#         lambda fn: second_by_basename.get(os.path.basename(fn))
#     )
#
#     # Merge into the SAME images_df/annots_df run_agent() reads from, so nothing else in
#     # the notebook needs to change -- track these ids separately so they're never confused
#     # with the DENTEX holdout_ids set (a different dataset isn't the same thing as a
#     # held-out slice of this one).
#     second_dataset_ids = set(second_images_df["id"])
#     images_df = pd.concat([images_df, second_images_df], ignore_index=True)
#     annots_df = pd.concat([annots_df, second_annots_df], ignore_index=True)
#
#     cross_dataset_results = run_agent_batch(
#         list(second_dataset_ids), agent_fn=run_agent,
#         cache_path=os.path.join(DATA_DIR, "cross_dataset_eval.json"),
#     )
#     cross_dataset_metrics = compute_evaluation_metrics(cross_dataset_results)
#     print(json.dumps(cross_dataset_metrics, indent=2, default=str))
# else:
#     print("Set SECOND_DATASET_PATH above, then re-run this cell.")

print(f"SECOND_DATASET_PATH is currently: {SECOND_DATASET_PATH!r} -- fill it in and "
      f"uncomment the block above once you've located the dataset.")


## 28. Offline self-test suite

108 cells and growing is large enough that a renamed function or a broken import can slip in
silently. This runs everything that needs no GPU, no dataset download, and no API key --
not exhaustive (nothing that touches the actual model or a live API is in scope here by
design), but it catches the cheapest class of bug fastest: the one that's easiest to
introduce during an edit. Needs sections 6, 11, 13, 14, 21, and 24 already defined, so it can
only run near the end of the notebook, not at the top.


In [ ]:
def run_offline_self_tests():
    results = {}

    try:
        synth = make_synthetic_dental_image()
        _ = tool_zoom_crop(synth.convert("RGB"), [500, 250, 100, 100])
        _ = tool_enhance_contrast(synth)
        results["deterministic tools (section 6)"] = "pass"
    except Exception as e:
        results["deterministic tools (section 6)"] = f"FAIL: {e}"

    try:
        assert tool_fdi_label(3, 6) == "36"
        assert tool_fdi_label(9, 1) is None
        results["FDI numbering (section 11)"] = "pass"
    except Exception as e:
        results["FDI numbering (section 11)"] = f"FAIL: {e}"

    try:
        parsed = parse_agent_json('{"tool": "zoom_crop", "args": {"bbox": [1, 2, 3, 4]}}')
        assert parsed is not None and parsed.get("tool") == "zoom_crop"
        assert parse_agent_json("not json at all") is None
        results["output JSON parsing (section 13)"] = "pass"
    except Exception as e:
        results["output JSON parsing (section 13)"] = f"FAIL: {e}"

    try:
        fake_traj = {
            "final_answer": {"quadrant": 1, "tooth_position": 6, "diagnosis": "Caries"},
            "format_ok": True, "tool_calls": 1,
            "turns": [{"parsed": {"tool": "zoom_crop"}, "tool_ok": True}],
        }
        gt = {"quadrant": 1, "tooth_position": 6, "diagnosis": "Caries"}
        _, components = combine_reward(fake_traj, gt)
        assert components["accuracy"] == 1.0
        results["reward function (section 14)"] = "pass"
    except Exception as e:
        results["reward function (section 14)"] = f"FAIL: {e}"

    try:
        assert flip_quadrant(1) == 2 and flip_quadrant(3) == 4
        assert abs(compute_iou([0, 0, 10, 10], [0, 0, 10, 10]) - 1.0) < 1e-6
        assert compute_iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0.0
        results["box geometry: flip_quadrant, IoU (section 24)"] = "pass"
    except Exception as e:
        results["box geometry: flip_quadrant, IoU (section 24)"] = f"FAIL: {e}"

    try:
        mean_diff, (lo, hi) = bootstrap_paired_diff_ci([1.0] * 10, [1.0] * 10)
        assert abs(mean_diff) < 1e-6 and abs(lo) < 1e-6 and abs(hi) < 1e-6
        results["bootstrap CI (section 21)"] = "pass"
    except Exception as e:
        results["bootstrap CI (section 21)"] = f"FAIL: {e}"

    print("Offline self-test results:")
    for name, status in results.items():
        marker = "OK  " if status == "pass" else "FAIL"
        detail = "" if status == "pass" else f"  -- {status}"
        print(f"  [{marker}] {name}{detail}")

    n_pass = sum(1 for s in results.values() if s == "pass")
    print(f"\n{n_pass}/{len(results)} passed.")
    return results


run_offline_self_tests()


## 29. Export a standalone inference module

Everything so far runs as notebook cells. This extracts the reusable, dataset-independent
inference-time pieces (deterministic tools, schema parsing, the agent loop, the reward
function) into one plain `.py` file — a starting point for a demo script or a small API,
once the agent loop feels solid enough to use outside the notebook.

**Deliberately not included**: the oracle grounding tool (it reads DENTEX's own ground
truth, which doesn't exist outside this notebook), the frozen `AGENT_SYSTEM_PROMPT` text
(it currently describes the oracle as an available tool, which would be misleading once
the oracle isn't there), and any training code (SFT/GRPO/Aim-1 only make sense inside this
notebook's environment, with its persistent cache and checkpoint directories). The exported
file's own header spells out exactly what to wire up before it's genuinely standalone.


In [ ]:
import inspect

EXPORT_SYMBOLS = [
    "tool_zoom_crop", "tool_enhance_contrast", "tool_fdi_label",
    "parse_agent_json", "generate_agent_reply", "run_agent",
    "reward_format", "reward_tool_validity", "reward_efficiency", "reward_accuracy", "combine_reward",
]

STANDALONE_HEADER_LINES = [
    "# Standalone inference-time dental agent, extracted from the DENTEX starter notebook.",
    "#",
    "# NOT included: the oracle grounding tool (tool_locate_abnormal_teeth) -- it reads",
    "# DENTEX's own ground-truth annotations, which don't exist outside this notebook's",
    "# environment. Register a real grounding tool (e.g. a trained Stage-0 detector) with",
    "# register_tool() before this will actually locate abnormal teeth:",
    "#",
    "#   register_tool(",
    "#       'locate_abnormal_teeth',",
    "#       lambda image_id: your_real_grounding_function(image_id),",
    "#       'locate_abnormal_teeth(): find and label abnormal teeth in the current image.',",
    "#   )",
    "#",
    "# Then rebuild the system prompt from whatever's actually registered -- do not reuse",
    "# the notebook's own AGENT_SYSTEM_PROMPT text verbatim, since it currently describes",
    "# the oracle tool that isn't included here:",
    "#",
    "#   AGENT_SYSTEM_PROMPT = (",
    "#       'You are a dental radiograph analysis agent. ...'  # copy the notebook's wording",
    '#       + "\\n".join(f"- {v[\'description\']}" for v in TOOLS.values())',
    "#   )",
    "#",
    "# Also requires `model` and `processor` loaded the same way the notebook's section 7",
    "# does -- this file has the logic, not a model-loading call, since which backbone or",
    "# checkpoint to load is a per-deployment choice.",
    "",
    "import json",
    "import re",
    "import torch",
    "from PIL import Image",
    "from qwen_vl_utils import process_vision_info",
    "",
    "TOOLS = {}",
    "",
    "",
    "def register_tool(name, fn, description):",
    '    TOOLS[name] = {"fn": fn, "description": description}',
    "",
]


def export_standalone_agent_module(path=None):
    """inspect.getsource() on each symbol in EXPORT_SYMBOLS -- works for normal top-level
    notebook-cell functions. If a symbol shows up as missing, check that the cell defining
    it has actually been run in this session."""
    path = path or os.path.join(DATA_DIR, "dental_agent.py")

    chunks = ["\n".join(STANDALONE_HEADER_LINES)]
    missing = []
    for name in EXPORT_SYMBOLS:
        obj = globals().get(name)
        if obj is None:
            missing.append(name)
            continue
        try:
            chunks.append(inspect.getsource(obj))
        except (TypeError, OSError) as e:
            missing.append(f"{name} ({e})")

    with open(path, "w") as f:
        f.write("\n\n".join(chunks))

    print(f"Wrote {path} ({len(EXPORT_SYMBOLS) - len(missing)}/{len(EXPORT_SYMBOLS)} symbols exported).")
    if missing:
        print(f"Could not export: {missing}")
    print(
        "\nRead the file's own header before using it -- it does NOT include a working "
        "grounding tool or a loaded model, on purpose (see the header comment for why)."
    )
    return path


print("export_standalone_agent_module() ready -- run it once the agent loop feels solid "
      "and you want a plain .py file to build a demo or API around.")


## 30. Diagnosis-predicting specialist detector (§6, baseline #5)

Another named gap: Stage 0's detector (section 24) only localizes and names a tooth --
`diagnosis` is always `None` there, on purpose (§5.3's division of labor). But §6's baseline
#5 -- "the 'why not just fine-tune a detector' comparison reviewers will ask for" -- needs a
detector that actually predicts *diagnosis*, so there's something to compare the agent's
diagnosis accuracy against. This trains one directly on the diagnosis labels, reusing most
of section 24's machinery (`build_stage0_detector`, `detection_collate_fn`).


In [ ]:
class DentexDiagnosisDetectionDataset(Dataset):
    """Same shape as section 24's DentexDetectionDataset, but labels each box by
    DIAGNOSIS class instead of FDI position -- what baseline #5 actually needs."""

    def __init__(self, images_df_in, annots_df_in, diag_col_in, categories_df_in):
        self.image_ids = sorted(annots_df_in["image_id"].unique())
        self.images_lookup = images_df_in.set_index("id")
        self.annots_df = annots_df_in
        self.diag_col = diag_col_in
        cat_lookup = dict(zip(categories_df_in["id"], categories_df_in["name"])) if diag_col_in else {}
        self.cat_lookup = cat_lookup
        present_names = sorted(annots_df_in[diag_col_in].map(cat_lookup).dropna().unique()) if diag_col_in else []
        self.class_to_idx = {name: i + 1 for i, name in enumerate(present_names)}  # 0 = background
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        row = self.images_lookup.loc[image_id]
        image = Image.open(row["local_path"]).convert("RGB")
        anns = self.annots_df[self.annots_df["image_id"] == image_id]

        boxes, labels = [], []
        for _, ann in anns.iterrows():
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0:
                continue
            diag_name = self.cat_lookup.get(ann.get(self.diag_col))
            if diag_name not in self.class_to_idx:
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(self.class_to_idx[diag_name])

        boxes_t = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32)
        labels_t = torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64)
        target = {"boxes": boxes_t, "labels": labels_t, "image_id": torch.tensor([image_id])}
        image_tensor = torchvision.transforms.functional.to_tensor(image)
        return image_tensor, target


def train_diagnosis_baseline_detector(epochs=1, batch_size=2, lr=5e-4, subset_n=None, verbose_every=10):
    """Plain supervised detector trained directly on diagnosis labels -- same
    non-held-out boundary as everything else in this notebook."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    train_ids = set(images_df[~images_df["id"].isin(holdout_ids)]["id"])
    train_annots = annots_df[annots_df["image_id"].isin(train_ids)]

    dataset = DentexDiagnosisDetectionDataset(
        images_df[images_df["local_path"].notna()], train_annots, diag_col, categories_df
    )
    if subset_n:
        dataset.image_ids = dataset.image_ids[:subset_n]
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=detection_collate_fn)

    detector = build_stage0_detector(num_classes=len(dataset.class_to_idx) + 1).to(device)
    optimizer = torch.optim.AdamW([p for p in detector.parameters() if p.requires_grad], lr=lr)

    detector.train()
    for epoch in range(epochs):
        running_loss, n_batches = 0.0, 0
        for step, (images, targets) in enumerate(loader):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = detector(images, targets)
            loss = sum(loss_dict.values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            n_batches += 1
            if (step + 1) % verbose_every == 0:
                print(f"  epoch {epoch + 1} step {step + 1}/{len(loader)}  loss={loss.item():.3f}")
        print(f"Epoch {epoch + 1}/{epochs} done -- mean loss: {running_loss / max(n_batches, 1):.3f}")

    detector.class_to_idx, detector.idx_to_class = dataset.class_to_idx, dataset.idx_to_class
    return detector


def evaluate_diagnosis_baseline_detector(detector, image_ids, score_threshold=0.5):
    """Diagnosis accuracy for the baseline detector -- its single highest-confidence
    prediction per image, compared on diagnosis only (this baseline doesn't predict
    quadrant/tooth, so it's judged on the one field it shares with the agent's own answer)."""
    correct = total = 0
    device = next(detector.parameters()).device
    detector.eval()
    for image_id in image_ids:
        gt_rows = tool_locate_abnormal_teeth(image_id)
        if not gt_rows:
            continue
        row = images_df[images_df["id"] == image_id].iloc[0]
        image = Image.open(row["local_path"]).convert("RGB")
        image_tensor = torchvision.transforms.functional.to_tensor(image).to(device)

        with torch.no_grad():
            prediction = detector([image_tensor])[0]

        total += 1
        if len(prediction["scores"]) == 0 or prediction["scores"][0] < score_threshold:
            continue
        pred_diag = detector.idx_to_class.get(int(prediction["labels"][0].item()))
        if pred_diag and pred_diag.lower() == str(gt_rows[0]["diagnosis"]).lower():
            correct += 1

    accuracy = correct / total if total else 0.0
    print(f"Diagnosis-baseline detector accuracy: {accuracy:.3f} ({correct}/{total})")
    return {"accuracy": accuracy, "n": total}


print("train_diagnosis_baseline_detector() / evaluate_diagnosis_baseline_detector() ready. "
      "Example:\n"
      "  diag_detector = train_diagnosis_baseline_detector(epochs=1, subset_n=20)\n"
      "  evaluate_diagnosis_baseline_detector(diag_detector, list(holdout_ids)[:20])")


## Next steps

This notebook now implements the full proposal loop, start to finish: data pipeline, tool
suite (with both an oracle stand-in and a real, trainable Stage-0 detector), the agent loop,
the reward function, a scalable Aim-1 trace pipeline, SFT, a GRPO skeleton, a complete
evaluation harness (batch running, metrics, majority baseline, H1/H2 ablations with a
bootstrap CI, R_judge, and a trajectory visualizer), paper-shaped results reporting, and a
cross-dataset generalization slot ready for a path. Still ahead, roughly in order:

- **Validate `grpo_step` for real**: run the smoke test in section 18 on 1-2 images with
  `group_size=2`, confirm `validate_span_alignment` returns `True` on a few different
  trajectories (not just the one demo case), try `epochs_per_batch=2` and confirm the ratio
  actually moves away from 1 on the second epoch (proof the clipping term is doing something,
  not a no-op), and watch actual GPU memory against section 7's estimate -- the reference-
  policy forward pass adds some overhead that estimate doesn't yet account for.
- **Train Stage 0 for real**: section 24's detector is a minimal sanity-check config (few
  epochs, small backbone, no LR schedule) -- once it trains cleanly on a tiny subset, scale up
  epochs/data, run `evaluate_stage0_detector` on `holdout_ids` for a real precision/recall/F1
  number, and check `visualize_detector_predictions` on several held-out images before ever
  swapping `tool_locate_abnormal_teeth_learned` in for the oracle in the agent loop's tool
  registry.
- **Run the section 16 scale-up for real**: `estimate_aim1_api_calls` then `run_aim1_batch`
  across the full non-held-out pool (not just the 2-example pilot), now that both exist.
- **Scale up the evaluation harness**: sections 19-22/26 default to tiny samples for a
  reason -- once SFT/GRPO checkpoints exist, run the full suite on the complete held-out
  evaluation set (`holdout_ids`), not a 5-10 image smoke test.
- **Run the reward-weight sweep for real**: `sweep_reward_weights` with `DEFAULT_WEIGHT_GRID`
  (or your own grid) on a larger sample, once there's a trained checkpoint whose behavior
  under different weightings is actually informative -- the base model's rollouts before any
  training won't show much signal here.
- **Move to TRL/EasyR1** once the hand-rolled GRPO here is understood and validated --
  production implementations still add distributed/multi-GPU rollout collection and more
  battle-tested numerics than this single-process reference version.
- **Cross-dataset generalization**: find the secondary dataset's real download location and
  fill in `SECOND_DATASET_PATH` in section 27 -- the loading/evaluation code is already
  written and waiting, commented out.
- **Run both baselines for real**: `run_zero_shot_baseline` (section 20) needs only an API
  key, no GPU or checkpoint; `train_diagnosis_baseline_detector` (section 30) needs the same
  free-tier GPU budget as Stage 0. Both are cheap relative to the agent training runs and
  give the final results table its two most-asked-for comparison points.

Save a checkpoint with `save_checkpoint(model, processor, tag=...)` at the end of each of
those stages so this cached setup carries forward.
